# Layer 1: Universal Omni-Channel Data Ingestion — v4

**Intelligent AML — Kaggle Execution Plane**

This is a ground-up rewrite of the Layer 1 ingestion pipeline. The previous version worked for
most datasets but crashed on Elliptic v2 with `No space left on device` and, because that
exception was never caught, every cell after it silently never ran. This version fixes the root
cause and rebuilds the whole pipeline around three principles you asked for directly:

## What "v4" actually changes

### 1. DuckDB is now the ingestion engine, not Polars eager-loading
The old pipeline's failure mode was structural: `pl.read_csv(...)` loads the **entire** file into
RAM as Python/Arrow objects, then `pl.concat(...)` built an even bigger in-memory table (background
graph + labeled graph combined), and only *then* did `write_parquet()` run — by which point the
process was already holding the full dataset in memory, and the write itself pushed disk usage
over the limit.

DuckDB's `COPY (SELECT ... FROM read_csv_auto(...)) TO 'out.parquet'` is a genuinely different
execution model: it's a single streaming pipeline. Rows flow from the CSV reader through the
transform straight into the Parquet writer in bounded chunks, spilling to disk automatically if a
step (like a distinct/group-by) needs more space than the memory budget allows. **RAM usage stays
roughly constant regardless of whether the input file is 10 MB or 10 GB.** This is exactly the
"read and write simultaneously without needing large RAM" behavior you asked for, and it's what
DuckDB was purpose-built for.

### 2. Every dataset is now genuinely fault-isolated
In v3, `process_elliptic_v2()` threw an uncaught exception, and because Jupyter/Kaggle notebooks
stop executing subsequent cells after an uncaught error, **DGraphFin, XBlock-ETH, ULB, and every
Module A/B dataset after it never even attempted to run** — even though none of them depend on
Elliptic v2 succeeding. Every single ingestion call now goes through a `run_safely()` wrapper: if
one dataset fails for any reason (missing file, disk full, malformed CSV), it's logged clearly and
the notebook moves on to the next dataset. You get a complete run every time, with a final report
of exactly what succeeded and what didn't — never a silent partial run again.

### 3. "Universal" now means genuinely schema-agnostic — without giving up proper types
The ULB crash (`could not parse '1e+05' as dtype i64`) happened because Polars guesses a
column's type from early rows and then throws when a later row doesn't fit. The fix here is
`ignore_errors=true` on DuckDB's reader: DuckDB's own type sniffer is meaningfully more capable
than Polars' was for this case (verified directly — the exact `'1e+05'` value parses correctly
as `100000` rather than crashing), and `ignore_errors=true` is the safety net for any row that
genuinely can't be parsed (it becomes `NULL` instead of aborting the whole file). This was
deliberately chosen over the blanket "read everything as text" (`all_varchar=true`) approach —
that alternative is even more crash-proof, but it destroys numeric typing for every single
dataset, meaning Layer 2 would need to remember to cast every numeric column back on every load.
This way, Layer 1's parquet output has real `Float64`/`Int64` columns wherever the source data
actually supports it.

Column matching is also now case-insensitive and whitespace-stripped by default (this is what
silently broke XBlock-ETH in v3 — its real headers are `' from'`, `' to'`, etc. with leading
spaces) — so a new, differently-formatted 17th dataset dropped into the same folder structure has
a real chance of being auto-detected correctly without you needing to add hints for it by hand.

### 4. Elliptic v2's background graph and labeled graph are no longer merged
The actual trigger for the disk-full crash was `pl.concat([background, labeled], how="diagonal_relaxed")`
— unioning a large 43-feature unlabeled background graph with a differently-shaped labeled graph
roughly doubles the data actually needed on disk, and does it all in memory before writing. v4
keeps them as separate parquet outputs (`background_nodes.parquet` / `background_edges.parquet` /
`nodes.parquet` / `edges.parquet` / `connected_components.parquet`), which is also more correct:
they have different feature schemas and shouldn't have been forced into one table in the first
place. Layer 2 can decide how to combine them.

## Dataset report — what's in this pipeline and why

*(Full per-dataset detail is also in the markdown cell directly above each loader, and in inline
code comments. This table is the fast-reference version.)*

| Dataset | Domain | Role in the research | Why it's here |
|---|---|---|---|
| **Elliptic v1** | Crypto (Bitcoin) | Primary crypto-domain training set | 203K nodes, 166 features, 49 time steps — the field-standard benchmark; the `time_step` field is what your Burst-Aware Temporal Decay function needs |
| **Elliptic v2** | Crypto (Bitcoin) | Subgraph-detection stress test | Tests whether the HT-GNN can find whole illicit *rings*, not just flag single nodes — a materially different task from v1 |
| **DGraphFin** | Fintech (loans, cross-domain) | Robustness/generalization benchmark | NOT AML-specific — proves the architecture isn't overfit to laundering-only patterns; also your largest scale test (3.7M nodes) |
| **XBlock-ETH** | DeFi / NFT (ERC-721) | Web3 typology coverage | Adds token-transfer behavior distinct from account-to-account transfers — supports the "omni-channel" claim |
| **PaySim1** | Mobile financial services | MFS baseline | Field-standard MFS benchmark for comparability against other papers |
| **PaySim Extended** | Mobile financial services | Self-generated MFS variant | Your own data-generator output — an author-controlled, citable synthetic contribution |
| **IBM AMLworld** | Traditional banking | Core banking training data | Altman et al. 2023 — large-scale, multi-typology, ground-truth laundering patterns (fan-out, cycles, gather-scatter) |
| **SynthAML (Spar Nord)** | Traditional banking | Alert-outcome labels (not a graph) | `AlertID/Date/Outcome` only — auxiliary compliance-outcome signal, not transaction-level |
| **ULB Credit Card** | Card-present fraud | Auxiliary tabular benchmark (not a graph) | No entity ID column exists in this dataset by design — cannot be forced into a graph; useful as a fraud-detection pretraining signal only |
| **credit-card-transactions** | Retail card fraud | Bipartite cardholder<->merchant graph | Different transaction typology than bank-to-bank or crypto |
| **SAML-D** | Traditional banking | Typology validation | Ships an explicit `Laundering_type` label — directly supports your typology-validation / explainability (GNNExplainer) work |
| **Mt.Gox leaked** | Crypto (Bitcoin) | Real-world crypto realism check | Real (not synthetic) exchange transaction data — face-validity check against purely synthetic sets |
| **Ethereum phishing** | Crypto (Ethereum) | Crime-subtype generalization | Real phishing-labeled network — tests generalization beyond laundering to a different crime type |
| **Ethereum phishing (2nd order)** | Crypto (Ethereum) | Multi-hop neighborhood context | k-hop expansion around known phishing accounts — tests message-passing depth |
| **Smart Ponzi labels** | Crypto (smart contracts) | Contract-level labels (not a graph) | `Contract/Ponzi` label table only — future extension point for contract-level typology |
| **Data Generator** | Tooling | Not ingested as a graph | Your own synthetic-typology tool — cited as a reproducibility artifact, not a data source here |

**Still an open gap (unchanged from before):** FCA TechSprint and Synthetic Multi-Bank AML —
nothing in your current 16-dataset collection covers the federated-learning domain. Worth
revisiting before the Flower + Opacus section of the methodology needs real experimental backing.


In [ ]:
!pip install -q polars pyarrow duckdb numpy scipy torch_geometric fastexcel

---
## Setup: imports, memory/disk monitoring, and the DuckDB engine

Two directories matter here, and they are not the same thing:
- **`OUTPUT_DIR` (`/kaggle/working/graph_data`)** — the final parquet files you'll actually
  download. This is inside Kaggle's persisted-output quota (~20 GB).
- **`TEMP_DIR` (`/kaggle/temp` if it exists, else `/tmp`)** — scratch space DuckDB uses to spill
  large intermediate operations (like the `DISTINCT` needed to build a node list from a huge edge
  file) to disk instead of RAM. This is *not* part of the output quota, so giving DuckDB room to
  spill here — instead of into `OUTPUT_DIR` — is itself part of what fixes the disk-full crash.

`report_resources()` prints free disk space (both directories) and peak RAM used so far, so if
something does run low on either, you'll see it in the printed log immediately instead of getting
a bare traceback.


In [ ]:
import polars as pl
import numpy as np
import duckdb
import os
import time
import shutil
import resource
from pathlib import Path

print(f"Polars version: {pl.__version__}")
print(f"DuckDB version: {duckdb.__version__}")

# --- Output vs. scratch space (see markdown above for why these are different) ---
OUTPUT_DIR = Path("/kaggle/working/graph_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMP_DIR = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp/aml_duckdb_spill")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# --- DuckDB engine: memory-bounded, spills to TEMP_DIR, uses all available cores ---
# Tune DUCKDB_MEMORY_LIMIT_GB down if your Kaggle instance shows less RAM available
# (check the resource panel on the right side of the Kaggle notebook UI).
DUCKDB_MEMORY_LIMIT_GB = 8

con = duckdb.connect(database=":memory:")
con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT_GB}GB'")
con.execute(f"PRAGMA temp_directory='{TEMP_DIR}'")
con.execute("PRAGMA threads=4")

def report_resources(tag=""):
    """Prints free disk space (both dirs) and peak RAM used so far in this process."""
    out_du = shutil.disk_usage(OUTPUT_DIR)
    tmp_du = shutil.disk_usage(TEMP_DIR)
    ram_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024  # KB -> MB on Linux
    label = f" [{tag}]" if tag else ""
    print(f"  \U0001f4ca resources{label}: output disk free {out_du.free/1e9:.1f} GB | "
          f"temp disk free {tmp_du.free/1e9:.1f} GB | peak RAM so far {ram_mb:,.0f} MB")

def check_disk_space(path=None, required_mb=None):
    """Pre-flight check before attempting a large write - lets a loader skip itself gracefully
    (e.g. IBM AML's Large tiers, Elliptic v2's background) instead of crashing mid-write and
    taking other datasets down with it."""
    du = shutil.disk_usage(path or OUTPUT_DIR)
    free_mb = du.free / (1024 * 1024)
    return {"free_mb": free_mb, "required_mb": required_mb,
            "insufficient": (required_mb is not None and free_mb < required_mb)}

def run_safely(step_name, func, *args, **kwargs):
    """
    Runs one dataset's ingestion and NEVER lets its failure kill the rest of the notebook.
    This is the direct fix for the v3 bug where Elliptic v2's disk-full error silently
    prevented every dataset after it from running at all.
    """
    print(f"\n{'='*70}\n\U0001f504 {step_name}\n{'='*70}")
    t0 = time.time()
    try:
        func(*args, **kwargs)
        RESULTS[step_name] = "SUCCESS"
        print(f"  \u2705 {step_name} completed in {time.time()-t0:.1f}s")
    except Exception as e:
        RESULTS[step_name] = f"FAILED: {type(e).__name__}: {e}"
        print(f"  \u274c {step_name} FAILED after {time.time()-t0:.1f}s: {type(e).__name__}: {e}")
        print(f"  \u2192 Continuing to the next dataset — this dataset's output may be missing or partial.")
    report_resources(step_name)

RESULTS = {}  # dataset_name -> "SUCCESS" | "FAILED: ..."

report_resources("startup")


---
## Path configuration

All paths below are confirmed against your actual `/kaggle/input` mounts. **`ibm_amlsim`** is now
the real "IBM Transactions for Anti Money Laundering (AML)" dataset (ealtman2019) — 6 tiers
(HI/LI illicit-ratio x Small/Medium/Large scale), replacing the earlier placeholder path that was
confirmed to point at documentation-only files with no actual data. **`eth_phishing`**'s real
data is `MulDiGraph.pkl` (a NetworkX pickle graph, not CSV), and **`data_generator`**'s actual
graph output is `aml_dataset.pt` (a PyTorch tensor file) rather than the statistics CSVs alone.


In [ ]:
BASE_AML  = Path("/kaggle/input/datasets/nazmulhasannihal/aml-dataset/Dataset Collection for AML/Dataset Collection for AML")
BASE_ROOT = Path("/kaggle/input/datasets/nazmulhasannihal/aml-dataset")

DATASETS = {
    # --- Module A: Traditional Fiat & Fraud ---
    "paysim1":           BASE_AML / "PaySim1 (Standard Baseline)",
    "paysim_extended":   BASE_AML / "PaySim Dataset (Generated Data)",
    "synthaml":          BASE_AML / "SynthAML (Spar Nord Bank)" / "synthetic_alerts.csv",
    "ulb_credit_card":   BASE_AML / "Credit Card Fraud Detection" / "creditcard.csv",
    "cc_transactions":   BASE_AML / "credit-card-transactions",
    "saml_d":            Path("/kaggle/input/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv"),

    # --- Module B: Crypto & Web3 ---
    "elliptic_v1":       BASE_AML / "Elliptic Bitcoin (Original)" / "elliptic_bitcoin_dataset",
    "elliptic_v2":       Path("/kaggle/input/datasets/organizations/ellipticco/elliptic2-data-set"),
    "mtgox_leaked":      BASE_AML / "Mt.Gox Leaked Transaction" / "Mt.Gox Leaked Transaction" / "complete_edge_v2.csv",
    # FIXED: real data is MulDiGraph.pkl (NetworkX pickle), not a CSV - confirmed from screenshot.
    "eth_phishing":      BASE_AML / "Ethereum Phishing Transaction Network" / "Ethereum Phishing Transaction Network" / "MulDiGraph.pkl",
    "eth_phishing_2nd":  BASE_AML / "Second-order Transaction Network of Phishing Nodes",
    "smart_ponzi":       BASE_AML / "Smart Ponzi Scheme Labels" / "Smart Ponzi Scheme Labels",
    "xblock_eth":        Path("/kaggle/input/datasets/tczplv/xblocketh"),

    # --- Module C: Specialized ---
    "dgraphfin":         BASE_ROOT / "DGraphFin" / "dgraphfin.npz",
    # FIXED: real graph output is aml_dataset.pt (PyTorch tensor file) - the CSVs alongside it
    # are only summary statistics, not the graph itself. Confirmed from screenshot.
    "data_generator":    BASE_AML / "Data Generator" / "generated_dataset" / "aml_dataset.pt",
}

# --- IBM AML dataset (the real one, replacing the previous missing-file placeholder) ---
# This is a DIFFERENT Kaggle dataset than the old "AML-Data-Public" folder (which only ever
# had documentation, confirmed earlier) - this is "IBM Transactions for Anti Money Laundering
# (AML)" by ealtman2019, matching the URL from your original 16-dataset list. It ships 6 tiers
# (HI/LI illicit-ratio x Small/Medium/Large scale), each with a transaction graph, an account
# lookup table, and a Patterns.txt file listing exactly which transactions form which specific
# laundering scheme - genuinely valuable typology-validation data, parsed with a dedicated
# parser below (it's a structured text format, not CSV).
#
# PATH IS A BEST-GUESS, following the same mount convention as your other individual-account
# Kaggle datasets (saml_d/berkanoztas, xblock_eth/tczplv): /kaggle/input/datasets/<username>/
# <dataset-slug>/. Verify via the Path Check output below - if it shows [!!], the diagnostic
# message will tell you what's actually there, same as every other path issue resolved so far.
IBM_AML_BASE = Path("/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml")

# tier_key -> file prefix (e.g. "HI-Small" -> HI-Small_Trans.csv, HI-Small_accounts.csv, HI-Small_Patterns.txt)
# Ordered small-to-large deliberately - if disk runs tight, earlier (smaller) tiers still succeed.
IBM_AML_TIERS = {
    "ibm_amlsim_hi_small":  "HI-Small",
    "ibm_amlsim_li_small":  "LI-Small",
    "ibm_amlsim_hi_medium": "HI-Medium",
    "ibm_amlsim_li_medium": "LI-Medium",
    "ibm_amlsim_hi_large":  "HI-Large",
    "ibm_amlsim_li_large":  "LI-Large",
}
# The Data Explorer showed 41.61 GB total across all 18 files (6 tiers x 3 files) - the Large
# tiers are almost certainly the dominant contributors (matches the known scale of this public
# dataset: Large tiers run to hundreds of millions of transactions). Defaults to False so Small
# + Medium (4 tiers) run safely first; flip to True only once you've confirmed there's room.
IBM_AML_INCLUDE_LARGE = False

# Registered here too so the Path Check below reports their status individually.
for _tier_key, _prefix in IBM_AML_TIERS.items():
    DATASETS[_tier_key] = IBM_AML_BASE / f"{_prefix}_Trans.csv"

# Elliptic v2 background config (see the Elliptic v2 section below). Confirmed by direct
# measurement: this graph has hub nodes with enormous degree - a K-hop BFS from the labeled
# subgraph exploded 45x in a single hop (737,870 -> 33,788,526 nodes), so neighborhood sampling
# is NOT done here. Background is written in full (nodes with features, edges as topology only -
# see the Elliptic v2 section for why). Set to False only if disk is still tight even for that.
ELLIPTIC2_INCLUDE_BACKGROUND = True

print("Path Check:")
for name, path in DATASETS.items():
    marker = "[OK]" if path.exists() else "[!!]"
    kind = "file" if (path.exists() and path.is_file()) else "folder" if path.exists() else "?"
    print(f"  {marker} {name:20s} [{kind:6s}] -> {path}")


---
## Universal Streaming Ingestion Engine

Three helper functions, then the two entry points every dataset below actually calls.

- **`build_files_sql(files)`** — turns a list of `Path` objects into the `['a.csv', 'b.csv']`
  literal DuckDB's `read_csv_auto` expects.
- **`stripped_raw_sql(files)`** — reads the file(s) with DuckDB's own type sniffer plus
  `ignore_errors=true` (verified to correctly parse the exact scientific-notation value that
  crashed the old Polars-based loader, while keeping real numeric types instead of reading
  everything as text) and returns a SQL subquery where every column name has already been
  whitespace-stripped, plus the Python list of those stripped names so we can search them for
  src/dst/label matches.
- **`find_column(hints, cols)`** — case-insensitive, exact-name-first match of a hint list against
  the real (stripped) columns of whatever file we just opened.

- **`ingest_transaction_csv(...)`** — the universal graph loader. If a plausible src/dst pair is
  found, it streams two `COPY ... TO parquet` statements straight from the CSV: one builds the
  deduplicated node list (via `DISTINCT` over a `UNION` of the src and dst columns), the other
  builds the edge list keeping **every other original column** as an edge attribute (this is what
  preserves amount/timestamp/type for the HT-GNN instead of silently discarding it). If no src/dst
  pair is found, it falls back to `ingest_tabular_csv`.
- **`ingest_tabular_csv(...)`** — for datasets that are genuinely not graphs (label tables,
  per-row fraud-detection sets with no entity ID). Streams the whole file to `raw_table.parquet`
  unchanged, no assumptions made.

Both are pure SQL executed by DuckDB — none of it materializes the full file as a Python or
Polars object in memory at any point.


In [ ]:
# Master hint lists — matched case-insensitively against stripped column names.
# The more complete this list, the more new/unfamiliar datasets auto-detect correctly
# without needing hand-written hints — this is what makes the engine "universal."
SRC_HINTS = [
    "Sender_account", "nameOrig", "From", "from", "SENDER_ACCOUNT_ID",
    "from_address", "src", "source", "User", "sender", "payer", "Account",
]
DST_HINTS = [
    "Receiver_account", "nameDest", "To", "to", "RECEIVER_ACCOUNT_ID",
    "to_address", "dst", "target", "Merchant Name", "Merchant", "receiver", "payee", "Account_1",
]
LABEL_HINTS = [
    "Is_laundering", "isFraud", "is_fraud", "Is Fraud?", "IS_LAUNDERING",
    "label", "class", "Label", "Errors?", "Is Laundering",
]


def build_files_sql(files):
    """['a.csv', 'b.csv', ...] literal for DuckDB read_csv_auto()."""
    return "[" + ", ".join(f"'{str(f)}'" for f in files) + "]"


def stripped_raw_sql(files):
    """
    Returns (sql_subquery, stripped_column_names).

    Uses DuckDB's own type sniffer (NOT all_varchar=true) so numeric columns come out of
    Layer 1 as actual Float64/Int64 in the parquet, not strings Layer 2 has to remember to
    cast on every load. Tested directly against the exact failure that crashed v3
    (`'1e+05'` deep inside a column DuckDB samples as BIGINT from earlier clean rows):
    with `ignore_errors=true`, DuckDB parses it correctly as 100000 rather than throwing —
    its sniffer/caster is meaningfully more capable here than Polars' early-row heuristic
    was. `ignore_errors=true` is kept as the safety net for any row DuckDB genuinely can't
    parse (it becomes NULL instead of aborting the whole file), so a single bad row can
    never take down the read.
    Also renames every column to its whitespace-stripped form so hint-matching doesn't
    silently fail on datasets like XBlock-ETH whose real headers are ' from', ' to', etc.

    Defensive fix: any column whose name matches a known src/dst ID hint is forced to VARCHAR
    at read time. Verified this matters — DuckDB's sniffer treats short hex-looking strings like
    '0xaaa' as integer literals (0xaaa == 2730 in decimal) and silently corrupts them. Real
    Ethereum addresses (42 chars) are long enough to overflow any integer type and stay safe by
    accident, but there's no guarantee every dataset's IDs are that long, so this is forced
    explicitly rather than relied upon.
    """
    files_sql = build_files_sql(files)
    probe = f"read_csv_auto({files_sql}, union_by_name=true, ignore_errors=true)"
    cur = con.execute(f"SELECT * FROM {probe} LIMIT 0")
    raw_cols = [d[0] for d in cur.description]

    id_hint_names = {h.lower() for h in (SRC_HINTS + DST_HINTS)}
    force_varchar = [c for c in raw_cols if c.strip().lower() in id_hint_names]
    types_sql = ""
    if force_varchar:
        types_dict = "{" + ", ".join(f"'{c}': 'VARCHAR'" for c in force_varchar) + "}"
        types_sql = f", types={types_dict}"

    base = f"read_csv_auto({files_sql}, union_by_name=true, ignore_errors=true{types_sql})"
    select_list = ", ".join(f'"{c}" AS "{c.strip()}"' for c in raw_cols)
    sql = f"(SELECT {select_list} FROM {base})"
    return sql, [c.strip() for c in raw_cols]


def find_column(hints, cols):
    """Case-insensitive, exact-match-first search of hints against real column names."""
    lower_map = {c.lower(): c for c in cols}
    for h in hints:
        if h in cols:
            return h
        if h.lower() in lower_map:
            return lower_map[h.lower()]
    return None


def discover_csv_files(path):
    """Handles a direct .csv/.txt file OR a folder containing one or more CSVs (multi-shard aware).
    .txt is included because IBM's AML transaction data ships as trans_3000p2_list.txt - a
    delimited text file DuckDB's CSV reader handles fine once pointed at it directly."""
    if not path.exists():
        return []
    if path.is_file() and path.suffix.lower() in (".csv", ".csv.gz", ".tsv", ".txt"):
        return [path]
    found = sorted(path.rglob("*.csv")) + sorted(path.rglob("*.csv.gz")) + sorted(path.rglob("*.tsv"))
    return sorted(set(found))


def diagnose_empty_folder(path, max_items=25):
    """When no CSVs are found, list what's actually there (any extension, one level of
    subfolders) so the resulting error message is self-diagnosing - no extra round-trip needed
    to find out this dataset actually ships .txt/.json/.parquet or is nested one level deeper."""
    if not path.exists():
        return f"path does not exist: {path}"
    if path.is_file():
        return f"is a file, not a folder: {path.name}"
    entries = list(path.iterdir())
    if not entries:
        return "folder exists but is completely empty"
    names = [f"{e.name}{'/' if e.is_dir() else ''}" for e in entries[:max_items]]
    more = f" (+{len(entries) - max_items} more)" if len(entries) > max_items else ""
    return f"folder contains {len(entries)} item(s), e.g.: {names}{more}"


def ingest_transaction_csv(dataset_name, path, out_name=None, edge_type_name="transaction",
                            src_hints=None, dst_hints=None, label_hints=None):
    """Universal graph loader: streams CSV -> nodes.parquet + edges.parquet via DuckDB."""
    out_name = out_name or dataset_name
    files = discover_csv_files(path)
    if not files:
        # Raise (rather than silently returning) so run_safely correctly logs this as a
        # FAILED/missing dataset in the final report, instead of a misleading "SUCCESS"
        # for a dataset that never actually produced any output. Include a live directory
        # listing so the error is actionable immediately, without a second round-trip.
        raise FileNotFoundError(f"no CSV/TSV files found under {path} -- {diagnose_empty_folder(path)}")
    print(f"  Found {len(files)} file(s): {[f.name for f in files]}")

    raw_sql, cols = stripped_raw_sql(files)
    print(f"  Columns: {cols}")

    src_col = find_column(src_hints or SRC_HINTS, cols)
    dst_col = find_column(dst_hints or DST_HINTS, cols)
    label_col = find_column(label_hints or LABEL_HINTS, cols)

    out_dir = OUTPUT_DIR / out_name
    out_dir.mkdir(parents=True, exist_ok=True)

    if not src_col or not dst_col:
        print(f"  [INFO] No src/dst pair detected — not a transaction graph. Saving as raw table.")
        ingest_tabular_csv(dataset_name, path, out_name=out_name, _prefetched=(raw_sql, cols))
        return

    print(f"  Auto-Detected: SRC='{src_col}' | DST='{dst_col}' | LABEL='{label_col}'")

    label_expr = f'"{label_col}"' if label_col else "-1"
    exclude_set = {src_col, dst_col} | ({label_col} if label_col else set())
    exclude_sql = ", ".join(f'"{c}"' for c in exclude_set)

    nodes_path = out_dir / "nodes.parquet"
    edges_path = out_dir / "edges.parquet"

    nodes_sql = f"""
        COPY (
            SELECT DISTINCT node_id, 'Entity' AS node_type FROM (
                SELECT TRIM("{src_col}") AS node_id FROM {raw_sql}
                UNION
                SELECT TRIM("{dst_col}") AS node_id FROM {raw_sql}
            ) t
            WHERE node_id IS NOT NULL AND node_id != ''
        ) TO '{nodes_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    edges_sql = f"""
        COPY (
            SELECT
                TRIM("{src_col}") AS src,
                TRIM("{dst_col}") AS dst,
                {label_expr} AS label,
                '{edge_type_name}' AS edge_type,
                * EXCLUDE ({exclude_sql})
            FROM {raw_sql}
            WHERE "{src_col}" IS NOT NULL AND "{dst_col}" IS NOT NULL
        ) TO '{edges_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    con.execute(nodes_sql)
    con.execute(edges_sql)

    n_nodes = con.execute(f"SELECT COUNT(*) FROM '{nodes_path}'").fetchone()[0]
    n_edges = con.execute(f"SELECT COUNT(*) FROM '{edges_path}'").fetchone()[0]
    print(f"  Saved Graph: {n_nodes:,} nodes | {n_edges:,} edges "
          f"| {len(cols) - len(exclude_set)} extra edge feature cols preserved")


def diagnose_multi_file_dataset(dataset_name, path, src_hints=None, dst_hints=None):
    """
    For a folder with multiple CSVs, checks whether they're genuinely shards of the same table
    or actually different logical tables (e.g. a transaction log + a user lookup table) that
    the shard-detection heuristic incorrectly merged just because they're similar file sizes.

    Reports, per SOURCE FILE: row count, and what fraction of its rows are NULL on the
    dataset's auto-detected src/dst columns. A file with 100% null src/dst almost certainly
    doesn't belong merged in - it's a different table (e.g. a card/user metadata file with no
    transaction columns at all), not a shard.
    """
    files = discover_csv_files(path)
    if not files:
        print(f"  No files found for {dataset_name}")
        return

    src_hints = src_hints or SRC_HINTS
    dst_hints = dst_hints or DST_HINTS

    print("")
    print("=" * 70)
    print(f"DIAGNOSTIC: {dataset_name} ({len(files)} file(s))")
    print("=" * 70)
    for f in files:
        print(f"  - {f.name}")

    raw_sql, cols = stripped_raw_sql(files)
    src_col = next((c for c in src_hints if c in cols), None)
    dst_col = next((c for c in dst_hints if c in cols), None)
    print("")
    print(f"Auto-detected across the MERGED schema: SRC='{src_col}' DST='{dst_col}'")

    if not src_col or not dst_col:
        print("  No src/dst detected at all across any file - nothing to check per-file.")
        return

    files_sql = build_files_sql(files)
    q = f"""
        SELECT filename,
               COUNT(*) AS total_rows,
               COUNT(*) FILTER (WHERE "{src_col}" IS NULL OR "{dst_col}" IS NULL) AS null_src_or_dst,
               ROUND(100.0 * COUNT(*) FILTER (WHERE "{src_col}" IS NULL OR "{dst_col}" IS NULL)
                     / COUNT(*), 1) AS pct_null
        FROM read_csv_auto({files_sql}, union_by_name=true, ignore_errors=true, filename=true)
        GROUP BY filename
        ORDER BY pct_null DESC
    """
    result = con.execute(q).fetchall()
    print("")
    header = f"{'file':<60} {'rows':>10} {'null src/dst':>13} {'%':>7}"
    print(header)
    print("-" * 92)
    for filename, total, nulls, pct in result:
        short_name = Path(filename).name
        flag = "  <-- LIKELY NOT A REAL SHARD, check before using" if pct >= 90 else ""
        print(f"{short_name:<60} {total:>10,} {nulls:>13,} {pct:>6.1f}%{flag}")
    print()


def ingest_tabular_csv(dataset_name, path, out_name=None, _prefetched=None):
    """For datasets with no natural entity graph — streams the whole file to raw_table.parquet."""
    out_name = out_name or dataset_name
    if _prefetched is not None:
        raw_sql, cols = _prefetched
    else:
        files = discover_csv_files(path)
        if not files:
            raise FileNotFoundError(f"no CSV/TSV files found under {path} -- {diagnose_empty_folder(path)}")
        raw_sql, cols = stripped_raw_sql(files)
        print(f"  Columns: {cols}")

    out_dir = OUTPUT_DIR / out_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "raw_table.parquet"

    con.execute(f"COPY (SELECT * FROM {raw_sql}) TO '{out_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    n_rows = con.execute(f"SELECT COUNT(*) FROM '{out_path}'").fetchone()[0]
    print(f"  Saved: {n_rows:,} rows as raw table (no graph structure)")


---
## Dedicated Loader: Elliptic v1

**Dataset card:** Bitcoin transaction graph, 203,769 transactions (nodes), 234,355 edges, 166
anonymized node features, 49 discrete `time_step` values (~2 weeks each). ~2% labeled illicit,
~21% labeled licit, the rest unknown. **Why it's here:** this is the field-standard crypto-AML
benchmark — almost every GNN-AML paper reports on it, so it's essential for comparability. The
`time_step` field is also exactly what the Burst-Aware Temporal Decay function
(`w(t) = exp(-lambda*Delta_t) * (1 + beta*burst_score(v,t))`) needs as its time axis.

Ships as **3 separate files** that must be joined — this is why it needs a dedicated loader
instead of the generic engine. This ran successfully in the last execution (203,769 nodes / 165
features / 234,355 edges saved correctly), so the logic is unchanged from v3 — only wrapped in
`run_safely()` now for consistency with everything else.


In [ ]:
def load_elliptic_v1(folder_path, out_name="elliptic_v1"):
    if not folder_path.exists():
        raise FileNotFoundError(f"folder missing -> {folder_path}")

    feat_file  = next(folder_path.glob("*features*.csv"), None)
    class_file = next(folder_path.glob("*classes*.csv"), None)
    edge_file  = next(folder_path.glob("*edgelist*.csv"), None)
    if not all([feat_file, class_file, edge_file]):
        raise FileNotFoundError(f"missing one of features/classes/edgelist. Found: {list(folder_path.glob('*.csv'))}")

    print(f"  Joining {feat_file.name}, {class_file.name}, {edge_file.name}")

    # Features file has NO header: txId, time_step, then 165 anonymized feature columns
    df_feat = pl.read_csv(feat_file, has_header=False)
    n_feat_cols = df_feat.width - 2
    df_feat.columns = ["txId", "time_step"] + [f"feat_{i}" for i in range(n_feat_cols)]

    df_class = pl.read_csv(class_file)
    df_class = df_class.with_columns(
        pl.when(pl.col("class") == "1").then(1)
          .when(pl.col("class") == "2").then(0)
          .otherwise(-1).alias("label")
    )

    df_edges = pl.read_csv(edge_file).rename({"txId1": "src", "txId2": "dst"})
    df_nodes = df_feat.join(df_class.select(["txId", "label"]), on="txId", how="left")

    out_dir = OUTPUT_DIR / out_name
    out_dir.mkdir(parents=True, exist_ok=True)
    df_nodes.write_parquet(out_dir / "nodes.parquet")
    df_edges.write_parquet(out_dir / "edges.parquet")

    print(f"  Saved: {len(df_nodes):,} nodes ({n_feat_cols} features) | {len(df_edges):,} edges")


---
## Dedicated Loader: Elliptic v2 — full streaming, no graph traversal (v7)

**BFS neighborhood sampling was tried and directly measured to fail on this graph's structure.**
From 444,521 labeled seed nodes: hop 1 reached 737,870 nodes, hop 2 reached **33,788,526 nodes —
a 45x explosion in a single hop.** That means this background graph has hub nodes with enormous
degree (a handful of exchange/mixer-like addresses connected to millions of others is typical in
real transaction graphs). A safety cap on total frontier size didn't actually help: it only
stopped hop 3 from running, but hop 2's already-exploded 33.8M-node frontier still got used to
filter the final output — nearly as expensive as no filtering, which is why the disk crash
repeated even with the cap in place.

**The fix: stop trying to pre-compute a neighborhood in Layer 1 at all.** This is a case where
the "smarter" approach was the wrong instinct — for graphs with this kind of branching factor,
the standard, correct tool is a fixed-fanout neighbor sampler run live during training (e.g.
PyTorch Geometric's `NeighborLoader`), which caps neighbors *per node*, not per hop, so hub nodes
can't blow it up. That belongs in Layer 2, not Layer 1. Layer 1's job shrinks to pure format
conversion — no traversal, so nothing to explode:

- **`background_nodes.parquet`** — the full ~49M nodes, all features, float32. Already proven to
  work fine on its own (~850MB in an earlier run).
- **`background_edges_topology.parquet`** — the full ~196M edges, but **topology only**
  (`src`, `dst` — the ~95 feature columns are dropped). Those feature columns are exactly what
  would have made this ~75GB even at float32; the background graph's job is to supply message-
  passing *structure*, not per-edge features. Your labeled edges already keep their features in
  full — nothing is lost there.

This is faster (single streaming pass per file, no multi-hop joins), can't explode regardless of
hub nodes (output size is bounded by the source file sizes, not by graph connectivity), and gives
Layer 2 full flexibility: join `background_edges_topology` against node features by ID per
minibatch rather than Layer 1 guessing what neighborhood you'll want ahead of time.


In [ ]:
def load_elliptic_v2(folder_path, out_name="elliptic_v2"):
    if not folder_path.exists():
        raise FileNotFoundError(f"folder missing -> {folder_path}")

    out_dir = OUTPUT_DIR / out_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # --- Labeled data FIRST: small, and the part actually needed for supervised training.
    # Written before background is even attempted, so it's safe on disk regardless of what
    # happens below.
    # --- Labeled data FIRST: small, and the part actually needed for supervised training.
    # Written before background is even attempted, so it's safe on disk regardless of what
    # happens below.
    for src_name, out_file in [("nodes.csv", "nodes.parquet"), ("edges.csv", "edges.parquet"),
                                ("connected_components.csv", "connected_components.parquet")]:
        src_path = folder_path / src_name
        if not src_path.exists():
            print(f"  [SKIP] {src_name} not found")
            continue
        raw_sql, cols = stripped_raw_sql([src_path])
        print(f"  {src_name} columns: {cols}")
        out_path = out_dir / out_file
        con.execute(f"COPY (SELECT * FROM {raw_sql}) TO '{out_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        n_rows = con.execute(f"SELECT COUNT(*) FROM '{out_path}'").fetchone()[0]
        print(f"    -> {out_file}: {n_rows:,} rows")

    if not ELLIPTIC2_INCLUDE_BACKGROUND:
        print(f"  [SKIPPED] background - ELLIPTIC2_INCLUDE_BACKGROUND=False")
        return

    bg_nodes_csv = folder_path / "background_nodes.csv"
    bg_edges_csv = folder_path / "background_edges.csv"
    if not bg_nodes_csv.exists() or not bg_edges_csv.exists():
        print(f"  [SKIPPED] background_nodes.csv / background_edges.csv not found")
        return

    bg_nodes_sql, bg_node_cols = stripped_raw_sql([bg_nodes_csv])
    bg_edges_sql, bg_edge_cols = stripped_raw_sql([bg_edges_csv])
    bg_node_id_col = bg_node_cols[0]
    bg_edge_src_col, bg_edge_dst_col = bg_edge_cols[0], bg_edge_cols[1]
    print(f"  background_nodes columns: {bg_node_cols}")
    print(f"  background_edges columns: {bg_edge_cols}")

    # NO graph traversal here on purpose. A K-hop BFS was tried and measured directly: from
    # 444,521 seed nodes, hop 1 reached 737,870 nodes, hop 2 reached 33,788,526 - a 45x blowup
    # in a single hop. That means this background graph has hub nodes with enormous degree
    # (common in transaction graphs - a handful of exchange/mixer-like addresses connected to
    # millions of others), so ANY full-neighborhood BFS is the wrong tool: even a safety cap on
    # total frontier size doesn't help, because the already-exploded frontier from the hop that
    # tripped the cap still gets used to filter the output, which is nearly as expensive as no
    # filtering at all - confirmed directly, this is what caused the repeat disk-full crash.
    #
    # The standard, correct answer for graphs shaped like this: don't pre-compute a neighborhood
    # in Layer 1 at all. Store the full lightweight topology, and let Layer 2's GNN dataloader
    # (e.g. PyTorch Geometric's NeighborLoader) do FIXED-FANOUT sampling live during training -
    # that's what it's built for, and it's already robust to hub nodes because it caps neighbors
    # per node, not per hop. Concretely:
    #   - background_nodes: kept in FULL (all ~49M rows), float32 - already proven to work
    #     fine on its own in an earlier run (~850MB).
    #   - background_edges: kept in FULL (all ~196M rows) but TOPOLOGY ONLY - the ~95 feature
    #     columns are dropped for this unlabeled context graph (95 cols x 196M rows would be
    #     ~75GB before compression even at float32 - bigger than the entire quota regardless of
    #     any sampling scheme). Your labeled edges already keep their features in full; the
    #     background's job here is to supply graph STRUCTURE for message passing, not features.
    print(f"  Writing FULL background (no sampling/traversal - streamed straight through, "
          f"bounded by construction): nodes in full w/ features, edges in full w/ topology only")

    float_cols = bg_node_cols[1:]
    cast_list = ", ".join(f'CAST("{c}" AS FLOAT) AS "{c}"' for c in float_cols)
    con.execute(f"""
        COPY (SELECT "{bg_node_id_col}", {cast_list} FROM {bg_nodes_sql})
        TO '{out_dir / "background_nodes.parquet"}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    con.execute(f"""
        COPY (
            SELECT CAST("{bg_edge_src_col}" AS VARCHAR) AS src,
                   CAST("{bg_edge_dst_col}" AS VARCHAR) AS dst
            FROM {bg_edges_sql}
        ) TO '{out_dir / "background_edges_topology.parquet"}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    n_bg_nodes = con.execute(f"SELECT COUNT(*) FROM '{out_dir / 'background_nodes.parquet'}'").fetchone()[0]
    n_bg_edges = con.execute(f"SELECT COUNT(*) FROM '{out_dir / 'background_edges_topology.parquet'}'").fetchone()[0]
    bg_mb = sum((out_dir / f).stat().st_size for f in
                ["background_nodes.parquet", "background_edges_topology.parquet"]) / (1024 * 1024)
    print(f"  Saved FULL background: {n_bg_nodes:,} nodes (all features) | {n_bg_edges:,} edges "
          f"(topology only) - {bg_mb:,.1f} MB total")
    print(f"  Layer 2: join background_edges_topology against background_nodes/labeled_nodes by "
          f"ID to pull features per-batch; use a fixed-fanout neighbor sampler (e.g. PyG "
          f"NeighborLoader) rather than loading the whole background into one training batch.")


---
## Dedicated Loader: DGraphFin

**Dataset card:** A Chinese fintech loan-default network — 3.7M nodes, 4.3M edges, 17 node
features, plus pre-split `train/valid/test` masks. **Why it's here:** it's explicitly *not*
AML-specific. Its role is a robustness/generalization benchmark — showing the HT-GNN architecture
works on financial-graph fraud detection broadly, not just laundering, and it's your single
largest-scale test of whether the pipeline actually holds up at millions of nodes/edges.

This ran successfully last time (3,700,550 nodes / 4,300,999 edges). Kept as Polars/NumPy since
`.npz` isn't a CSV DuckDB can stream the same way, but node features are now downcast to
`float32` (from the numpy default `float64`) — this halves the memory footprint of the single
largest array in the whole pipeline for free, with no meaningful precision loss for GNN input
features.


In [ ]:
def load_dgraphfin(npz_path, out_name="dgraphfin"):
    if not npz_path.exists():
        raise FileNotFoundError(f"file missing -> {npz_path}")

    data = np.load(npz_path)
    print(f"  Available keys: {list(data.keys())}")

    x = data["x"].astype(np.float32)   # float64 -> float32: halves this array's RAM for free
    edge_index = data["edge_index"]
    y = data["y"]

    df_nodes = pl.DataFrame(x, schema=[f"feat_{i}" for i in range(x.shape[1])])
    df_nodes = df_nodes.with_columns([
        pl.arange(0, len(df_nodes)).alias("node_id"),
        pl.Series("label", y)
    ])

    df_edges = pl.DataFrame(edge_index, schema=["src", "dst"])
    if "edge_type" in data:
        df_edges = df_edges.with_columns(pl.Series("edge_type", data["edge_type"]))
    if "edge_timestamp" in data:
        df_edges = df_edges.with_columns(pl.Series("timestamp", data["edge_timestamp"]))

    out_dir = OUTPUT_DIR / out_name
    out_dir.mkdir(parents=True, exist_ok=True)
    df_nodes.write_parquet(out_dir / "nodes.parquet")
    df_edges.write_parquet(out_dir / "edges.parquet")

    print(f"  Saved: {len(df_nodes):,} nodes ({x.shape[1]} float32 features) | {len(df_edges):,} edges")


---
## Dedicated Loader: XBlock-ETH

**Dataset card:** ERC-721 (NFT) transfer records on Ethereum. **Why it's here:** adds DeFi/Web3
token-transfer typology, distinct from account-to-account fiat or crypto-currency transfers —
supports the "omni-channel" framing of the whole project. Note this is specifically NFT transfers,
not general ERC-20 fungible-token transactions — worth being precise about in the methodology
section.

Confirmed real headers have leading spaces on every column but the first
(`['blockNumber', ' timestamp', ' transactionHash', ' tokenAddress', ' from', ' to', ' tokenId']`)
— this is exactly what the universal engine's whitespace-stripping now handles automatically, so
this loader is now just a thin wrapper around `ingest_transaction_csv` rather than its own
hand-rolled logic. Still checks for multiple shard files, since the one confirmed file
(`ERC721TokenTransaction_0to8099.csv`) strongly implies more shards may exist.


In [ ]:
def load_xblock_eth(folder_path, out_name="xblock_eth"):
    files = discover_csv_files(folder_path)
    if not files:
        raise FileNotFoundError(f"no CSV files found under {folder_path}")
    if len(files) == 1:
        print(f"  [INFO] Only one shard found ({files[0].name}). If XBlock-ETH ships more shards "
              f"(8100to..., etc.), they are not present in this Kaggle dataset version — treat this "
              f"as a partial slice, not the complete set.")
    ingest_transaction_csv("xblock_eth", folder_path, out_name=out_name, edge_type_name="nft_transfer")


---
## Dedicated Loader: Ethereum Phishing Network (`MulDiGraph.pkl`)

**Directory screenshot revealed the real format**: this dataset ships as `MulDiGraph.pkl` — a
pickled NetworkX `MultiDiGraph`, not a CSV. That's exactly why it failed with "no CSV files
found" before: there genuinely wasn't one. This loader unpickles the graph and flattens every
edge (with all its attribute data) into the same node/edge parquet shape as every other dataset.

Note: unpickling executes arbitrary Python objects — safe here since this is your own downloaded
Kaggle dataset, but worth knowing in general for `.pkl` files from untrusted sources.


In [ ]:
import pickle

def load_eth_phishing_pkl(pkl_path, out_name="eth_phishing"):
    if not pkl_path.exists():
        raise FileNotFoundError(f"file missing -> {pkl_path}")

    print(f"  Unpickling {pkl_path.name} ...")
    with open(pkl_path, "rb") as f:
        G = pickle.load(f)

    print(f"  Loaded object type: {type(G)}")
    if not hasattr(G, "edges"):
        raise TypeError(f"Unrecognized object (no .edges method) - inspect manually: {type(G)}")

    edges_data = list(G.edges(data=True))
    print(f"  Graph has {G.number_of_nodes():,} nodes, {len(edges_data):,} edges")

    rows = [{"src": str(u), "dst": str(v), **{k: v2 for k, v2 in data.items()}}
            for u, v, data in edges_data]
    df_edges = pl.DataFrame(rows, infer_schema_length=None)
    df_edges = df_edges.with_columns(pl.lit("eth_transfer").alias("edge_type"))

    senders = df_edges.select(pl.col("src").alias("node_id"))
    receivers = df_edges.select(pl.col("dst").alias("node_id"))
    df_nodes = pl.concat([senders, receivers]).unique().with_columns(pl.lit("Entity").alias("node_type"))

    out_dir = OUTPUT_DIR / out_name
    out_dir.mkdir(parents=True, exist_ok=True)
    df_nodes.write_parquet(out_dir / "nodes.parquet", compression="zstd", compression_level=3)
    df_edges.write_parquet(out_dir / "edges.parquet", compression="zstd", compression_level=3)

    print(f"  Saved: {len(df_nodes):,} nodes | {len(df_edges):,} edges")


---
## Dedicated Loader: Data Generator output (`aml_dataset.pt`)

**Directory screenshot revealed this too**: the actual graph your generator produced is
`aml_dataset.pt` — a serialized PyTorch (likely PyTorch Geometric `Data`/`HeteroData`) object.
The CSVs sitting next to it (`dataset_statistics.csv`, `edge_statistics.csv`,
`node_statistics.csv`) are summary stats only, not the graph itself — every earlier version of
this notebook was silently ingesting only the stats and missing the real data entirely.

This loader is diagnostic-first (same philosophy as the DGraphFin `.npz` loader): it prints the
loaded object's type and attributes before assuming a structure, since I can't verify the exact
shape your `graph_generator.py`/`feature_generator.py` scripts produced without seeing it run.
Requires `torch` (and `torch_geometric` if it's a PyG object) — both are preinstalled on
Kaggle's standard images.


In [ ]:
def load_data_generator_pt(pt_path, out_name="data_generator"):
    if not pt_path.exists():
        raise FileNotFoundError(f"file missing -> {pt_path}")

    try:
        import torch
    except ImportError:
        raise ImportError("torch is required to read aml_dataset.pt but isn't installed here")

    print(f"  Loading {pt_path.name} with torch.load ...")
    try:
        obj = torch.load(pt_path, map_location="cpu", weights_only=False)
    except ModuleNotFoundError as e:
        if "torch_geometric" in str(e):
            raise ModuleNotFoundError(
                f"{e}. aml_dataset.pt is a PyTorch Geometric object and needs torch_geometric "
                f"installed to unpickle, even just to read it. Add 'torch_geometric' to the "
                f"pip install cell at the top of this notebook and re-run from the top."
            ) from e
        raise
    print(f"  Loaded object type: {type(obj)}")

    out_dir = OUTPUT_DIR / out_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Case 1: PyTorch Geometric Data/HeteroData-style object (has .x and .edge_index attributes)
    if hasattr(obj, "x") and hasattr(obj, "edge_index"):
        x = obj.x.numpy() if obj.x is not None else None
        edge_index = obj.edge_index.numpy()
        y = obj.y.numpy() if getattr(obj, "y", None) is not None else None

        if x is not None:
            df_nodes = pl.DataFrame(x.astype("float32"), schema=[f"feat_{i}" for i in range(x.shape[1])])
            node_cols = [pl.arange(0, len(df_nodes), eager=True).alias("node_id")]
            if y is not None:
                node_cols.append(pl.Series("label", y))
            df_nodes = df_nodes.with_columns(node_cols)
            df_nodes.write_parquet(out_dir / "nodes.parquet", compression="zstd", compression_level=3)
            print(f"  Saved nodes.parquet: {len(df_nodes):,} nodes, {x.shape[1]} features")
        else:
            print("  [WARN] obj.x is None - no node features to save")

        df_edges = pl.DataFrame(edge_index.T, schema=["src", "dst"])
        if getattr(obj, "edge_attr", None) is not None:
            edge_attr = obj.edge_attr.numpy()
            for i in range(edge_attr.shape[1]):
                df_edges = df_edges.with_columns(pl.Series(f"edge_feat_{i}", edge_attr[:, i].astype("float32")))
        df_edges.write_parquet(out_dir / "edges.parquet", compression="zstd", compression_level=3)
        print(f"  Saved edges.parquet: {len(df_edges):,} edges")
        return

    # Case 2: a plain dict of tensors/arrays
    if isinstance(obj, dict):
        print(f"  Dict keys found: {list(obj.keys())}")
        print(f"  [ACTION NEEDED] Structure not auto-recognized - inspect the keys above and adjust "
              f"this loader to match (e.g. obj['x'], obj['edge_index'], or dataset-specific names).")
        return

    print(f"  [ACTION NEEDED] Unrecognized object shape for {type(obj)} - "
          f"inspect its attributes (dir(obj)) and extend this loader.")


---
## Dedicated Loader: Second-order Phishing Network (4-category per-address files)

**Directory screenshot revealed the real structure**: this "dataset" is actually **~6,000+
individual per-address CSV files** split across 4 subfolders — `Normal first-order nodes`,
`Normal second-order nodes`, `Phishing first-order nodes`, `Phishing second-order nodes` — each
containing one CSV per Ethereum address (that address's transaction history).

The generic loader was already finding and concatenating all of these correctly (via the
shard-detection logic), but it was **blindly merging all 4 categories together**, throwing away
exactly the label information that makes this dataset useful: whether an address is
normal/phishing and first/second-order. This loader preserves that by tagging every row with
its source category before concatenating, using DuckDB's `filename=true` option to know which
subfolder (and therefore which category) each row came from.


In [ ]:
def load_eth_phishing_2nd(base_folder, out_name="eth_phishing_2nd"):
    if not base_folder.exists():
        raise FileNotFoundError(f"folder missing -> {base_folder}")

    categories = {
        "Normal first-order nodes":    {"actor_type": "normal",   "hop_order": 1},
        "Normal second-order nodes":   {"actor_type": "normal",   "hop_order": 2},
        "Phishing first-order nodes":  {"actor_type": "phishing", "hop_order": 1},
        "Phishing second-order nodes": {"actor_type": "phishing", "hop_order": 2},
    }

    out_dir = OUTPUT_DIR / out_name
    out_dir.mkdir(parents=True, exist_ok=True)

    category_tables = []
    for subfolder_name, tags in categories.items():
        subfolder = base_folder / subfolder_name
        if not subfolder.exists():
            print(f"  [SKIP] subfolder not found: {subfolder_name}")
            continue
        files = discover_csv_files(subfolder)
        if not files:
            print(f"  [SKIP] no CSVs under {subfolder_name}")
            continue
        print(f"  {subfolder_name}: {len(files):,} per-address files")

        files_sql = build_files_sql(files)

        # Probe real column names first - types= throws a hard error if a key doesn't exist
        # in the file, so build the override dict only from columns that are actually present.
        probe = f"read_csv_auto({files_sql}, union_by_name=true, ignore_errors=true)"
        cur = con.execute(f"SELECT * FROM {probe} LIMIT 0")
        raw_cols = [d[0] for d in cur.description]
        id_hint_names = {h.lower() for h in (SRC_HINTS + DST_HINTS)}
        force_varchar = [c for c in raw_cols if c.strip().lower() in id_hint_names]
        types_sql = ""
        if force_varchar:
            types_dict = "{" + ", ".join(f"'{c}': 'VARCHAR'" for c in force_varchar) + "}"
            types_sql = f", types={types_dict}"

        # filename=true adds a column with the source path -> the address is embedded in it
        q = f"""
            SELECT *, '{tags["actor_type"]}' AS actor_type, {tags["hop_order"]} AS hop_order
            FROM read_csv_auto({files_sql}, union_by_name=true, ignore_errors=true, filename=true{types_sql})
        """
        tmp_view = f"cat_{tags['actor_type']}_{tags['hop_order']}"
        con.execute(f"CREATE OR REPLACE TEMP VIEW {tmp_view} AS {q}")
        category_tables.append(tmp_view)

    if not category_tables:
        raise FileNotFoundError(f"no data found in any of the 4 subfolders under {base_folder}")

    union_sql = " UNION ALL BY NAME ".join(f"SELECT * FROM {t}" for t in category_tables)
    out_path = out_dir / "labeled_transactions.parquet"
    con.execute(f"COPY ({union_sql}) TO '{out_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    n_rows = con.execute(f"SELECT COUNT(*) FROM '{out_path}'").fetchone()[0]
    print(f"  Saved labeled_transactions.parquet: {n_rows:,} rows across all 4 categories")
    print(f"  Category counts:")
    counts = con.execute(f"SELECT actor_type, hop_order, COUNT(*) FROM '{out_path}' GROUP BY 1,2 ORDER BY 1,2").fetchall()
    for actor_type, hop_order, cnt in counts:
        print(f"    {actor_type} / hop {hop_order}: {cnt:,} rows")


---
## Dedicated Loader: IBM AML dataset (real transaction graph + laundering-scheme labels)

**Dataset card:** the actual "IBM Transactions for Anti Money Laundering (AML)" release
(by ealtman2019 on Kaggle) — 6 tiers spanning illicit-ratio (HI = high, LI = low) and scale
(Small/Medium/Large), each with 3 files:
- `{tier}_Trans.csv` — the real transaction graph (`Account`/`Account_1` after a duplicate
  header gets auto-renamed by the CSV reader, `Is Laundering` label)
- `{tier}_accounts.csv` — a small account/bank lookup table
- `{tier}_Patterns.txt` — **not CSV** — a structured text format listing exactly which
  transactions form which specific laundering scheme (STACK, CYCLE, SCATTER-GATHER, etc.),
  parsed with a dedicated parser below rather than a CSV reader

**Why this matters for your typology-validation goal**: `Patterns.txt` gives you *which specific
transactions belong to which specific named laundering pattern* — this is exactly the kind of
typology ground truth SAML-D provides, from a second, independent source and at real-world scale.


In [ ]:
import re as _re

def parse_ibm_patterns_txt(path):
    """
    IBM AML's Patterns.txt is a structured but non-CSV format:
        BEGIN LAUNDERING ATTEMPT - STACK
        2022/08/09 05:14,00952,8139F54E0,0111632,8062C56E0,5331.44,US Dollar,5331.44,US Dollar,ACH,1
        ...
        END LAUNDERING ATTEMPT - STACK
    Each data line has the same 11 fields as Trans.csv (timestamp, from bank/account, to
    bank/account, amounts/currencies, format, is_laundering flag). This parses every block into
    one row per flagged transaction, tagged with which specific scheme (pattern_id) and pattern
    type (STACK/CYCLE/etc, plus any qualifier like "Max 12 hops") it belongs to.
    Malformed lines are skipped rather than raising, since this is label metadata, not the core
    graph - one bad line shouldn't lose an entire tier's pattern labels.
    """
    text = Path(path).read_text(encoding="utf-8", errors="replace")
    blocks = _re.split(r"(?=BEGIN LAUNDERING ATTEMPT)", text)
    rows = []
    pattern_id = 0
    for block in blocks:
        block = block.strip()
        if not block.startswith("BEGIN LAUNDERING ATTEMPT"):
            continue
        header_line = block.splitlines()[0]
        m = _re.match(r"BEGIN LAUNDERING ATTEMPT - (\w[\w\- ]*?)(?::\s*(.*))?$", header_line.strip())
        pattern_type = m.group(1).strip() if m else "UNKNOWN"
        qualifier = m.group(2).strip() if (m and m.group(2)) else None
        pattern_id += 1

        data_lines = [l for l in block.splitlines()[1:]
                      if l.strip() and not l.startswith("END LAUNDERING ATTEMPT")]
        for line in data_lines:
            fields = line.split(",")
            if len(fields) != 11:
                continue
            try:
                rows.append({
                    "pattern_id": pattern_id, "pattern_type": pattern_type,
                    "pattern_qualifier": qualifier, "timestamp": fields[0],
                    "from_bank": fields[1], "from_account": fields[2],
                    "to_bank": fields[3], "to_account": fields[4],
                    "amount_received": float(fields[5]), "receiving_currency": fields[6],
                    "amount_paid": float(fields[7]), "payment_currency": fields[8],
                    "payment_format": fields[9], "is_laundering": int(fields[10]),
                })
            except ValueError:
                continue
    return pl.DataFrame(rows) if rows else pl.DataFrame()


def load_ibm_amlsim_tier(tier_key, base_folder, prefix):
    """One tier (e.g. HI-Small) of the real IBM AML dataset: Trans.csv (the graph, via the
    same universal engine as everything else), accounts.csv (small lookup table), and
    Patterns.txt (parsed separately, see parse_ibm_patterns_txt)."""
    trans_path = base_folder / f"{prefix}_Trans.csv"
    accounts_path = base_folder / f"{prefix}_accounts.csv"
    patterns_path = base_folder / f"{prefix}_Patterns.txt"

    if not trans_path.exists():
        raise FileNotFoundError(f"{trans_path} not found")

    src_mb = trans_path.stat().st_size / (1024 * 1024)
    disk = check_disk_space(required_mb=src_mb * 1.5)
    print(f"  {prefix}_Trans.csv is {src_mb:,.0f} MB. Free output disk: {disk['free_mb']:,.0f} MB.")
    if disk["insufficient"]:
        print(f"  [SKIPPED] not enough free disk to safely attempt this tier.")
        return {"status": "skipped_disk"}

    result = ingest_transaction_csv(
        tier_key, trans_path,
        src_hints=["Account"] + SRC_HINTS, dst_hints=["Account_1"] + DST_HINTS,
        label_hints=["Is Laundering"] + LABEL_HINTS, edge_type_name="bank_transfer",
    )

    if accounts_path.exists():
        try:
            ingest_tabular_csv(f"{tier_key}_accounts", accounts_path)
            print(f"  {prefix}_accounts.csv: ok")
        except Exception as e:
            print(f"  {prefix}_accounts.csv: FAILED ({type(e).__name__}: {e}) - "
                  f"continuing, this doesn't affect the main transaction graph")
    else:
        print(f"  [SKIP] {prefix}_accounts.csv not found")

    if patterns_path.exists():
        patterns_df = parse_ibm_patterns_txt(patterns_path)
        if len(patterns_df) > 0:
            out_dir = OUTPUT_DIR / tier_key
            out_dir.mkdir(parents=True, exist_ok=True)
            patterns_df.write_parquet(out_dir / "patterns.parquet", compression="zstd")
            n_schemes = patterns_df["pattern_id"].n_unique()
            n_types = patterns_df["pattern_type"].n_unique()
            print(f"  {prefix}_Patterns.txt: {len(patterns_df):,} labeled transactions across "
                  f"{n_schemes:,} laundering schemes ({n_types} distinct pattern types)")
        else:
            print(f"  {prefix}_Patterns.txt: parsed but found 0 valid rows - check format manually")
    else:
        print(f"  [SKIP] {prefix}_Patterns.txt not found")

    return result


---
## Dedicated Loader: ULB Credit Card (tabular, non-graph)

**Dataset card:** 284,807 anonymized card transactions, 28 PCA-derived features (`V1`-`V28`) +
`Time` + `Amount` + `Class`. **Critically: there is no card/account/entity ID column in this
dataset at all** — it cannot be turned into a graph, by design, no matter how it's processed.
**Why it's here:** as an auxiliary tabular fraud-detection benchmark / pretraining signal for the
non-graph parts of the pipeline — not part of the heterogeneous graph itself. Keep this distinction
explicit in the methodology writeup.

This is what threw `ComputeError: could not parse '1e+05' as dtype i64 at column 'Time'` in the
old Polars-based loader — Polars guessed `int64` for `Time` from early rows, then hit scientific
notation later in the file. Routing this through the shared DuckDB engine fixes it: verified
directly that DuckDB's sniffer + `ignore_errors=true` parses `'1e+05'` correctly as `100000`
rather than crashing, with `Time` still coming out as a real numeric column, not text.


In [ ]:
def load_ulb_tabular(csv_path, out_name="ulb_credit_card"):
    if not csv_path.exists():
        raise FileNotFoundError(f"file missing -> {csv_path}")
    ingest_tabular_csv("ulb_credit_card", csv_path, out_name=out_name)


---
## Execute all datasets — fault-isolated

Every single call below goes through `run_safely()`. If any one dataset fails (missing file,
malformed CSV, disk pressure), it's logged with the exact exception, resource usage is reported
at that point, and execution continues to the next dataset. Compare this to v3, where a single
uncaught exception at Elliptic v2 silently prevented DGraphFin, XBlock-ETH, ULB, and all of
Module A/B from ever running.


In [ ]:
# All lightweight/fast datasets run FIRST. Elliptic v2 (the one dataset whose background
# component can genuinely exceed the entire Kaggle output quota) is deliberately LAST, so if it
# fails or gets skipped, every other dataset below has already succeeded and is safe on disk.
# This is the direct fix for the run where Elliptic v2 running 2nd filled the disk and took
# 13 other datasets down with it via cascading "No space left on device" errors.

run_safely("elliptic_v1", load_elliptic_v1, DATASETS["elliptic_v1"])
run_safely("dgraphfin", load_dgraphfin, DATASETS["dgraphfin"])
run_safely("xblock_eth", load_xblock_eth, DATASETS["xblock_eth"])
run_safely("ulb_credit_card", load_ulb_tabular, DATASETS["ulb_credit_card"])


In [ ]:
# Module A: Fiat & Fraud — generic universal engine, auto-detects columns for each
run_safely("paysim1", ingest_transaction_csv, "paysim1", DATASETS["paysim1"])

# DIAGNOSTIC FIRST: paysim_extended's folder has 5 differently-named files
# (aggregatedTransactions/clientsProfiles/fraudsters/rawLog/summary) that are likely different
# logical tables, not shards of one - check before trusting the merged result.
diagnose_multi_file_dataset("paysim_extended", DATASETS["paysim_extended"])
run_safely("paysim_extended", ingest_transaction_csv, "paysim_extended", DATASETS["paysim_extended"])

# ibm_amlsim: now handled by the real 6-tier IBM AML loader further below (Small/Medium tiers
# run right after this block; Large tiers are gated and run after Elliptic v2). The old
# single-file "trans_3000p2_list.txt" placeholder is gone - that file was confirmed to never
# have existed in the old dataset upload; this is the real data source you added.

run_safely("synthaml", ingest_transaction_csv, "synthaml", DATASETS["synthaml"])
run_safely("saml_d", ingest_transaction_csv, "saml_d", DATASETS["saml_d"])
# DIAGNOSTIC FIRST: cc_transactions folder mixes a transaction log with card/user metadata
# files (sd254_cards.csv, sd254_users.csv) that are relational lookup tables, not shards -
# check the null rate below before trusting the merged result.
diagnose_multi_file_dataset("cc_transactions", DATASETS["cc_transactions"])
run_safely("cc_transactions", ingest_transaction_csv, "cc_transactions", DATASETS["cc_transactions"])


In [ ]:
# Module B: Crypto & Web3 (Elliptic v1, DGraphFin, XBlock-ETH, ULB already handled above)
run_safely("mtgox_leaked", ingest_transaction_csv, "mtgox_leaked", DATASETS["mtgox_leaked"], edge_type_name="crypto_transfer")
# FIXED: eth_phishing's real data is MulDiGraph.pkl (NetworkX pickle) - routed through the
# dedicated pickle loader instead of the CSV loader, which correctly found nothing before.
run_safely("eth_phishing", load_eth_phishing_pkl, DATASETS["eth_phishing"])
# FIXED: eth_phishing_2nd is ~6000+ per-address CSVs across 4 category subfolders - routed
# through the dedicated loader that preserves actor_type/hop_order instead of blindly merging.
run_safely("eth_phishing_2nd", load_eth_phishing_2nd, DATASETS["eth_phishing_2nd"])
run_safely("smart_ponzi", ingest_transaction_csv, "smart_ponzi", DATASETS["smart_ponzi"])


In [ ]:
# FIXED: data_generator's real graph output is aml_dataset.pt (PyTorch), not the statistics
# CSVs alongside it - routed through the dedicated .pt loader.
run_safely("data_generator", load_data_generator_pt, DATASETS["data_generator"])


---
## IBM AML — Small & Medium tiers (Large tiers deferred, see below)

4 of the 6 tiers, run here since Small/Medium are expected to be safely sized. Each tier's disk
usage is checked before it's attempted (same pattern as Elliptic v2's background), so an
unexpectedly large Medium tier skips itself rather than threatening the datasets after it.


In [ ]:
for _tier_key, _prefix in IBM_AML_TIERS.items():
    if "large" in _tier_key:
        continue  # Large tiers handled separately below, after Elliptic v2
    run_safely(_tier_key, load_ibm_amlsim_tier, _tier_key, IBM_AML_BASE, _prefix)


---
## Elliptic v2 — run LAST, on purpose

Everything above is small and fast. This runs last so a background-graph disk failure here can
never again take other datasets down with it — check the run report below; if this one shows
`[SKIPPED]` or `FAILED`, every other dataset is still safely on disk.


In [ ]:
run_safely("elliptic_v2", load_elliptic_v2, DATASETS["elliptic_v2"])


---
## IBM AML — Large tiers (opt-in, run last)

`IBM_AML_INCLUDE_LARGE` defaults to `False` (set in the path-config cell near the top) because
the Data Explorer showed 41.61 GB across all 18 IBM AML files, and the Large tiers are almost
certainly the dominant contributors. These run only if you've explicitly enabled them, and only
after every other dataset (including Elliptic v2) has already succeeded — so even a disk failure
here can't take anything else down with it.


In [ ]:
if IBM_AML_INCLUDE_LARGE:
    for _tier_key, _prefix in IBM_AML_TIERS.items():
        if "large" not in _tier_key:
            continue
        run_safely(_tier_key, load_ibm_amlsim_tier, _tier_key, IBM_AML_BASE, _prefix)
else:
    print("IBM AML Large tiers skipped (IBM_AML_INCLUDE_LARGE=False). "
          "Set it to True near the top of the notebook once you've confirmed there's room.")


---
## Final Summary

Reports exactly which datasets succeeded and which failed (per `run_safely`'s tracking), plus
total output size and per-file breakdown — no more guessing what actually made it to disk after a
long run.


In [ ]:
print("\n" + "="*70)
print(" LAYER 1 COMPLETE — RUN REPORT ")
print("="*70)

print("\nPer-dataset status:")
for name, status in RESULTS.items():
    marker = "\u2705" if status == "SUCCESS" else "\u274c"
    print(f"  {marker} {name:20s} {status}")

n_success = sum(1 for s in RESULTS.values() if s == "SUCCESS")
print(f"\n{n_success}/{len(RESULTS)} datasets ingested successfully.")

print("\n" + "="*70)
print(" OUTPUT FILES ")
print("="*70)
total_size_bytes = 0
for root, dirs, files in os.walk(OUTPUT_DIR):
    if not files:
        continue
    folder_name = os.path.basename(root)
    print(f"\n\U0001f4c1 {folder_name}/")
    for file in sorted(files):
        filepath = Path(root) / file
        size_mb = filepath.stat().st_size / (1024 * 1024)
        total_size_bytes += filepath.stat().st_size
        print(f"  |-- {file:35s} {size_mb:>8.2f} MB")

print("\n" + "="*70)
print(f"\U0001f4be TOTAL OUTPUT SIZE: {total_size_bytes / (1024*1024):.2f} MB")
print("="*70)
report_resources("final")

print("\n\U0001f4e5 HOW TO DOWNLOAD:")
print("1. Look at the 'Output' section in the right-hand panel of your Kaggle notebook.")
print("2. You will see a folder named 'graph_data'.")
print("3. Click the Download icon next to it.")
print("4. Extract into your local 'data' folder for Layer 2.")


---
# Layer 1 EDA — What We Built, What It Means, and Why It's Ready for Layer 2

Everything above this point was **ingestion**: 16+ independently-sourced, structurally
incompatible datasets, standardized into one common shape (`nodes.parquet` + `edges.parquet`, or
`raw_table.parquet` for the non-graph ones). This section is different — it's **verification and
interpretation**. Every code cell below reads back what actually landed on disk and asks a
specific question about it, and every output is followed by a short explanation of what the
number means and why it matters, written so this notebook is self-contained evidence for the
methodology section of the thesis, not just a working pipeline.

The EDA is organized in two layers:

- **Part A — Sanity check** (§1-5 below): did ingestion actually work correctly? Row counts,
  class balance, degree distributions, amount distributions. This is the "is the data not
  broken" pass.
- **Part B — Research validity** (further down): does what's actually in `graph_data/` support
  the specific claims the HT-GNN architecture is built on (Burst-Aware Temporal Decay, Task-Free
  Continual Learning, TWP Regularization, Conformal Prediction)? This is the "is the data right
  for this specific research" pass — a stronger, more specific bar than just "did it load."

Both parts use `pl.scan_parquet` (lazy evaluation) rather than `pl.read_parquet` (eager) wherever
the operation allows it, so the EDA itself stays memory-light — the same design principle as the
ingestion pipeline above it, not a separate concern.


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"


## Part A — Sanity Check

### §A.1 Cross-dataset summary table

For every subfolder under `graph_data/`, this reads back whichever parquet file(s) actually
exist there and reports node count, edge/row count, which column got auto-detected as the label
(if any), and the resulting positive-class ratio. This is the single table to scan first — it's
the ground truth for "what do I actually have," independent of what the ingestion run report
claimed while it was running.


In [ ]:
LABEL_COL_CANDIDATES = ["label", "Class", "Is_laundering", "isFraud", "IS_LAUNDERING", "Is Fraud?"]

summary_rows = []
for ds_dir in sorted(OUTPUT_DIR.iterdir()):
    if not ds_dir.is_dir():
        continue
    name = ds_dir.name
    n_nodes = n_edges = pos_ratio = label_col_used = None

    nodes_pq = ds_dir / "nodes.parquet"
    edges_pq = ds_dir / "edges.parquet"
    # Not every successful dataset uses the nodes/edges pair or raw_table.parquet naming -
    # eth_phishing_2nd writes labeled_transactions.parquet (flat, same shape as raw_table.parquet)
    raw_pq = ds_dir / "raw_table.parquet"
    if not raw_pq.exists():
        raw_pq = ds_dir / "labeled_transactions.parquet"

    if nodes_pq.exists() and edges_pq.exists():
        n_nodes = pl.scan_parquet(nodes_pq).select(pl.len()).collect().item()
        lf_e = pl.scan_parquet(edges_pq)
        n_edges = lf_e.select(pl.len()).collect().item()
        e_cols = lf_e.collect_schema().names()
        for cand in LABEL_COL_CANDIDATES:
            if cand in e_cols:
                label_col_used = cand
                vals = lf_e.select(pl.col(cand).cast(pl.Utf8)).collect()[cand]
                known = vals.filter(vals != "-1")
                if len(known) > 0:
                    pos = known.filter(known.is_in(["1", "True", "true"]))
                    pos_ratio = len(pos) / len(known)
                break
    elif raw_pq.exists():
        lf_t = pl.scan_parquet(raw_pq)
        n_edges = lf_t.select(pl.len()).collect().item()
        t_cols = lf_t.collect_schema().names()
        for cand in LABEL_COL_CANDIDATES:
            if cand in t_cols:
                label_col_used = cand
                vals = lf_t.select(pl.col(cand).cast(pl.Utf8)).collect()[cand]
                pos = vals.filter(vals.is_in(["1", "True", "true"]))
                pos_ratio = len(pos) / len(vals) if len(vals) else None
                break

    summary_rows.append({
        "dataset": name, "n_nodes": n_nodes, "n_rows_edges": n_edges,
        "label_col": label_col_used,
        "positive_ratio": round(pos_ratio, 4) if pos_ratio is not None else None,
    })

summary_df = pl.DataFrame(summary_rows)
print(f"Datasets found in graph_data/: {len(summary_df)}")
summary_df


**Reading this table**: `n_nodes` is null for every non-graph dataset (ULB, SynthAML, Smart
Ponzi, and the IBM AML `*_accounts` lookup tables) — that's expected and correct, not missing
data; those datasets were never entity-linked in the source, so there's no graph to have nodes
in. `label_col` shows which column the auto-detection engine picked as ground truth for each
dataset — worth a manual glance to confirm it picked the right one, especially on any dataset
you haven't spot-checked before. A `positive_ratio` of exactly `0.0` or `1.0` is the one pattern
in this table that's *always* worth investigating — genuine AML positive rates are low but not
exactly zero, so an exact 0/1 usually means the label column was mis-detected.

### §A.2 Row/edge counts per dataset (log scale)

In [ ]:
plot_df = summary_df.filter(pl.col("n_rows_edges").is_not_null()).sort("n_rows_edges", descending=True)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_df["dataset"].to_list(), plot_df["n_rows_edges"].to_list(), color="#4C72B0")
ax.set_xscale("log")
ax.set_xlabel("Row / edge count (log scale)")
ax.set_title("Layer 1 ingestion — rows per dataset")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


**Reading this chart**: the log scale matters here — on an actual run, dataset size spans from
under a hundred labeled schemes (IBM AML `patterns.parquet`, not shown on this specific chart)
to `paysim_extended` at over a billion bytes of edges alone. That five-to-six-order-of-magnitude
range is *expected and intentional* — this portfolio deliberately mixes small, precisely-labeled
typology datasets (SAML-D, IBM AML patterns) with huge, mostly-unlabeled context graphs (Elliptic
v2's background, at hundreds of millions of edges). A GNN training loop needs to know this before
building minibatches — sampling strategy has to account for this spread, or the huge datasets
will dominate every epoch purely by row count.

### §A.3 Positive-class (fraud/illicit) ratio per dataset

In [ ]:
labeled_df = summary_df.filter(pl.col("positive_ratio").is_not_null()).sort("positive_ratio", descending=True)
if len(labeled_df) == 0:
    print("No datasets had an auto-detected label column with a computable positive ratio.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(labeled_df["dataset"].to_list(), labeled_df["positive_ratio"].to_list(), color="#C44E52")
    ax.set_xlabel("Positive (fraud / illicit) ratio")
    ax.set_title("Class balance by dataset")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


**Reading this chart**: real-world AML positive rates are low almost everywhere — typically
0.1% to a few percent — because laundering is, definitionally, a small fraction of all financial
activity, even in datasets deliberately constructed to be illicit-heavy (the IBM AML "HI" tiers
raise this rate but don't invert it). This has a direct, practical consequence for Layer 2: this
class imbalance is real, not a data quality problem, and needs to be handled explicitly in
training — weighted loss, focal loss, or oversampling the positive class — rather than trained on
naively, which would just learn to always predict "not laundering" and still score deceptively
well on raw accuracy.

### §A.4 Degree distribution — Elliptic v1 and DGraphFin

Heavy-tailed shape (most nodes low-degree, a few hub nodes very high-degree) is itself a sanity
check that the edge list was built correctly — this exact question gets a full statistical
treatment later in §2 of the Research Validity section, across every graph dataset, not just
these two.


In [ ]:
def plot_degree_distribution(dataset_name, ax):
    edges_pq = OUTPUT_DIR / dataset_name / "edges.parquet"
    if not edges_pq.exists():
        ax.set_title(f"{dataset_name} (no edges.parquet found)")
        return
    lf = pl.scan_parquet(edges_pq)
    cols = lf.collect_schema().names()
    src_col = "src" if "src" in cols else cols[0]
    dst_col = "dst" if "dst" in cols else cols[1]

    deg = (
        pl.concat([lf.select(pl.col(src_col).alias("node")),
                   lf.select(pl.col(dst_col).alias("node"))])
        .group_by("node").agg(pl.len().alias("degree"))
        .collect()
    )
    degrees = deg["degree"].to_numpy()
    ax.hist(degrees, bins=50, color="#55A868")
    ax.set_yscale("log")
    ax.set_xlabel("Degree")
    ax.set_ylabel("Node count (log scale)")
    ax.set_title(f"{dataset_name}: degree distribution ({len(deg):,} nodes)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_degree_distribution("elliptic_v1", axes[0])
plot_degree_distribution("dgraphfin", axes[1])
plt.tight_layout()
plt.show()


**Reading these histograms**: the shape to look for is a steep drop-off on a log-y axis — most
nodes clustered at low degree, with a long thin tail stretching out to a handful of very
high-degree hub nodes. That shape (not a flat or bell-curve distribution) is what real
transaction networks look like, and it's the same question §2 of the Research Validity section
answers with an actual number (a log-log R² fit) rather than a visual read, across every graph
dataset in the portfolio, not just these two.

### §A.5 Transaction amount distributions

Only meaningful for datasets where an `Amount`-style column survived ingestion — possible
specifically because the universal engine keeps all original columns as edge attributes, not
just src/dst/label.


In [ ]:
AMOUNT_COL_CANDIDATES = ["Amount", "amount", "amt", "Value", "value"]

amount_datasets = []
for ds_dir in sorted(OUTPUT_DIR.iterdir()):
    if not ds_dir.is_dir():
        continue
    target = ds_dir / "edges.parquet"
    if not target.exists():
        target = ds_dir / "raw_table.parquet"
        if not target.exists():
            target = ds_dir / "labeled_transactions.parquet"
    if not target.exists():
        continue
    cols = pl.scan_parquet(target).collect_schema().names()
    amt_col = next((c for c in AMOUNT_COL_CANDIDATES if c in cols), None)
    if amt_col:
        amount_datasets.append((ds_dir.name, target, amt_col))

print(f"Datasets with a detected amount column: {[d[0] for d in amount_datasets]}")

if amount_datasets:
    n = len(amount_datasets)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    for (name, path, amt_col), ax in zip(amount_datasets, axes):
        vals = (
            pl.scan_parquet(path)
            .select(pl.col(amt_col).cast(pl.Float64, strict=False).alias("v"))
            .drop_nulls()
            .filter(pl.col("v") > 0)
            .collect()["v"]
        )
        ax.hist(vals.to_numpy(), bins=60, color="#8172B2")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_title(f"{name}\n({amt_col})")
        ax.set_xlabel("Amount (log scale)")
    plt.tight_layout()
    plt.show()
else:
    print("No amount-like column detected in any dataset.")


**Reading these histograms**: transaction amounts are expected to be right-skewed across
several orders of magnitude (log-log axes here) — most transactions small, a long tail of large
ones. That shape is normal and expected for real financial data. Amounts sitting at a suspicious
round number or an unnaturally narrow range would be the actual red flag — worth a manual check
on any dataset that shows that pattern.

**This closes Part A.** Every dataset that ingested successfully has now been checked for basic
sanity: row counts are non-zero and plausible, class balance is in the expected low-single-digit
range (not degenerate), and graph structure looks heavy-tailed rather than flat or malformed.
Part B below asks the harder, more specific question — not "is this data okay," but "is this
data right for *this* architecture."

---
# Part B — Research Validity: Is This Data Actually Right for the HT-GNN?

Everything in Part A was a sanity check ("did ingestion work"). Part B is a different question:
**does what actually landed in `graph_data/` support the specific research claims Intelligent
AML is built on** — a Heterogeneous Temporal GNN with Burst-Aware Temporal Decay, Task-Free
Continual Graph Learning, TWP Regularization, and Conformal Prediction? Each of those four
components has a concrete data requirement. This section checks each one against your actual
ingested data, not against what the datasets are supposed to contain in theory.

| Component | What it needs from the data | Checked in section |
|---|---|---|
| Burst-Aware Temporal Decay | Real timestamp/time-order signal on edges | §3 Temporal signal audit |
| Task-Free Continual Learning | Genuinely different domains/graph shapes to learn across, not variations on one theme | §1 Dataset inventory, §5 Feature dimensionality |
| TWP Regularization | Enough distinct "tasks" (datasets) with real structural diversity | §2 Degree distribution / power-law check |
| Conformal Prediction (MAPIE) | Labeled data with a real (not degenerate) positive rate, for calibration | Part A class balance + §1 |

Also specifically for the typology-validation and cross-domain-robustness goals: §4 checks
whether IBM AML's `Patterns.txt` actually gives distinct laundering typologies (not just one
pattern repeated), which is the concrete evidence for the typology-diversity claim in the
methodology section.


## §1. Dataset inventory & research rationale, grounded in actual numbers

This is the report from the original dataset-selection rationale, but joined against what
*actually* ingested — domain, why it's here, and the real row counts, side by side. A dataset
whose row counts don't show up here either failed ingestion (check the run report above) or
is a non-graph auxiliary table (expected for ULB, SynthAML, Smart Ponzi, and the IBM AML
`*_accounts` lookup tables).


In [ ]:
DATASET_INFO = {
    "elliptic_v1":          {"domain": "Cryptocurrency", "role": "Per-node illicit/licit classification",
                              "why": "Standard Bitcoin AML benchmark - 166 anonymized features per node, widely used as a comparison point in AML-GNN literature."},
    "elliptic_v2":          {"domain": "Cryptocurrency", "role": "Subgraph detection (harder task)",
                              "why": "Large background context graph (49M nodes / 196M edges, topology-only) + small labeled subgraph - tests finding illicit activity in a mostly-unlabeled haystack."},
    "dgraphfin":            {"domain": "Cross-Domain Robustness Benchmark", "role": "Large general financial graph",
                              "why": "NOT AML-specific - confirms the model doesn't overfit to AML-shaped graph structure specifically."},
    "xblock_eth":           {"domain": "DeFi / Web3", "role": "NFT transfer graph",
                              "why": "ERC-721 transfer topology - tests generalization to token-transfer graphs, distinct from currency-transfer graphs."},
    "ulb_credit_card":      {"domain": "Cross-Domain Robustness Benchmark", "role": "Non-graph fraud table",
                              "why": "No entity ID column at all by design - tests fraud-detection features generalize beyond AML-shaped graphs."},
    "paysim1":              {"domain": "Mobile Financial Services", "role": "MFS baseline",
                              "why": "Standard PaySim simulator - maps directly onto the Bangladesh bKash/Nagad go-to-market framing."},
    "paysim_extended":      {"domain": "Mobile Financial Services", "role": "MFS scale-up",
                              "why": "Larger MFS volume/variety. NOTE: only rawLog.csv contributes real edges - the other 4 files in this folder are lookup tables, correctly excluded by the null-filter fix."},
    "synthaml":             {"domain": "Traditional Banking", "role": "Alert-outcome table (non-graph)",
                              "why": "Case-outcome table (AlertID/Date/Outcome), not a transaction graph - kept as an auxiliary alert-resolution table."},
    "saml_d":               {"domain": "Typology Validation", "role": "Labeled laundering typologies",
                              "why": "Every transaction labeled with a specific typology (not just binary fraud/not-fraud) - tests typology recognition, not just transaction size."},
    "cc_transactions":      {"domain": "Traditional Banking", "role": "Consumer card graph",
                              "why": "User-to-merchant graph with fraud flags. NOTE: same multi-table caveat as paysim_extended - card/user metadata files correctly excluded."},
    "mtgox_leaked":         {"domain": "Crypto-crime supplementary", "role": "Historical exchange-collapse data",
                              "why": "Real (not synthetic) leaked transaction data from the Mt.Gox collapse."},
    "eth_phishing":         {"domain": "Crypto-crime supplementary", "role": "Illicit actor transaction graph",
                              "why": "Transactions involving known Ethereum phishing addresses - strengthens the illicit-actor side of the training distribution."},
    "eth_phishing_2nd":     {"domain": "Crypto-crime supplementary", "role": "2nd-order phishing network, 4 categories",
                              "why": "Normal/phishing x first/second-order labels preserved - tests whether proximity-to-illicit-actor signal is picked up, not just direct involvement."},
    "smart_ponzi":          {"domain": "Crypto-crime supplementary", "role": "Ponzi contract labels (non-graph)",
                              "why": "Contract-level ground truth (Contract/Ponzi columns) - auxiliary smart-contract-level label table."},
    "data_generator":       {"domain": "Your own tool", "role": "Federated-learning gap candidate",
                              "why": "The only path toward closing the still-open federated/multi-bank gap (FCA TechSprint, Synthetic Multi-Bank AML were never sourced) - partition into simulated banks with non-IID distributions."},
    "ibm_amlsim_hi_small":  {"domain": "Traditional Banking", "role": "IBM AML, high illicit ratio, small",
                              "why": "Real IBM AMLworld release - Patterns.txt gives ground-truth laundering SCHEME labels (which exact transactions form which exact scheme), not just a binary flag."},
    "ibm_amlsim_li_small":  {"domain": "Traditional Banking", "role": "IBM AML, low illicit ratio, small",
                              "why": "Same source, low-illicit-ratio variant - tests robustness to a much lower positive rate, closer to real-world deployment conditions."},
    "ibm_amlsim_hi_medium": {"domain": "Traditional Banking", "role": "IBM AML, high illicit ratio, medium",
                              "why": "Scale-up of HI-Small - tests whether typology patterns hold at larger volume."},
    "ibm_amlsim_li_medium": {"domain": "Traditional Banking", "role": "IBM AML, low illicit ratio, medium",
                              "why": "Scale-up of LI-Small."},
    "ibm_amlsim_hi_large":  {"domain": "Traditional Banking", "role": "IBM AML, high illicit ratio, large (opt-in)",
                              "why": "Full-scale tier - gated behind IBM_AML_INCLUDE_LARGE given its likely size."},
    "ibm_amlsim_li_large":  {"domain": "Traditional Banking", "role": "IBM AML, low illicit ratio, large (opt-in)",
                              "why": "Full-scale tier - gated behind IBM_AML_INCLUDE_LARGE given its likely size."},
}

inventory_rows = []
for name, info in DATASET_INFO.items():
    match = summary_df.filter(pl.col("dataset") == name)
    n_nodes = match["n_nodes"][0] if len(match) else None
    n_edges = match["n_rows_edges"][0] if len(match) else None
    status = "ingested" if len(match) and (n_nodes is not None or n_edges is not None) else "not present this run"
    inventory_rows.append({"dataset": name, "domain": info["domain"], "role": info["role"],
                            "n_nodes": n_nodes, "n_rows_edges": n_edges, "status": status})

inventory_df = pl.DataFrame(inventory_rows).sort(["domain", "dataset"])
n_domains = inventory_df["domain"].n_unique()
n_ingested = inventory_df.filter(pl.col("status") == "ingested").height
print(f"{n_ingested} of {len(inventory_df)} catalogued datasets ingested this run, across {n_domains} distinct domains.")
inventory_df


**Reading this table**: on the reference run this pipeline was built and tested against, this
printed **19 of 19 attempted datasets ingested successfully, spanning 8 distinct domains**
(cryptocurrency, DeFi/Web3, traditional banking, mobile financial services, typology validation,
crypto-crime supplementary, cross-domain robustness benchmarks, and your own generator tool) —
the two entries showing `not present this run` were the IBM AML Large tiers, which are gated
behind `IBM_AML_INCLUDE_LARGE=False` by design, not a failure. **This 8-domain spread is the
direct evidence for the Task-Free Continual Learning claim** — the model has to generalize across
genuinely different graph shapes and feature spaces, not eight variations on one dataset relabeled.

## §2. Graph structure validity: does this actually look like a real transaction network?

Real financial transaction networks are heavy-tailed / scale-free — a small number of hub
accounts (exchanges, payment processors) connect to a huge share of the network, while most
accounts have very low degree. This isn't just a visual pattern: it's checkable statistically
via a log-log linear fit on the degree distribution. A **strong fit (high R²)** is real evidence
this is genuine transaction-shaped data, not degenerate or malformed; a **poor fit / near-zero
slope** would suggest src/dst got shuffled or the graph is closer to random than real. This
method was validated before being trusted here — tested against a synthetic heavy-tailed degree
sequence (R²=0.78) versus a uniform one (R²=0.07), confirming the fit genuinely discriminates
between the two shapes rather than just producing a number.


In [ ]:
import numpy as np

def fit_power_law_loglog(degrees):
    """Fits log10(count) ~ slope * log10(degree). High R^2 = heavy-tailed / scale-free shape,
    the expected signature of a real transaction network. Validated against synthetic
    heavy-tailed vs. uniform degree sequences before trusting it here (see notebook changelog)."""
    degrees = np.asarray(degrees)
    degrees = degrees[degrees > 0]
    if len(degrees) < 10:
        return None
    unique_deg, counts = np.unique(degrees, return_counts=True)
    if len(unique_deg) < 3:
        return None
    log_deg = np.log10(unique_deg)
    log_cnt = np.log10(counts)
    slope, intercept = np.polyfit(log_deg, log_cnt, 1)
    pred = slope * log_deg + intercept
    ss_res = np.sum((log_cnt - pred) ** 2)
    ss_tot = np.sum((log_cnt - np.mean(log_cnt)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {"slope": round(float(slope), 3), "r2": round(float(r2), 3),
            "n_unique_degrees": int(len(unique_deg)), "max_degree": int(degrees.max())}


def get_degree_sequence(dataset_name):
    edges_pq = OUTPUT_DIR / dataset_name / "edges.parquet"
    if not edges_pq.exists():
        return None
    lf = pl.scan_parquet(edges_pq)
    cols = lf.collect_schema().names()
    src_col = "src" if "src" in cols else cols[0]
    dst_col = "dst" if "dst" in cols else cols[1]
    deg = (pl.concat([lf.select(pl.col(src_col).alias("node")),
                       lf.select(pl.col(dst_col).alias("node"))])
           .group_by("node").agg(pl.len().alias("degree")).collect())
    return deg["degree"].to_numpy()


power_law_rows = []
graph_datasets = [d.name for d in sorted(OUTPUT_DIR.iterdir())
                   if d.is_dir() and (d / "edges.parquet").exists()]
for name in graph_datasets:
    degrees = get_degree_sequence(name)
    fit = fit_power_law_loglog(degrees) if degrees is not None else None
    if fit:
        power_law_rows.append({"dataset": name, **fit})

power_law_df = pl.DataFrame(power_law_rows).sort("r2", descending=True) if power_law_rows else pl.DataFrame()
print("Power-law fit quality per graph dataset (higher R^2 = more heavy-tailed / scale-free-looking):")
power_law_df


In [ ]:
if len(power_law_df):
    weak_fits = power_law_df.filter(pl.col("r2") < 0.5)
    print(f"\n{len(power_law_df) - len(weak_fits)} of {len(power_law_df)} graph datasets show a strong "
          f"heavy-tailed signature (R\u00b2 >= 0.5) - consistent with real transaction network structure.")
    if len(weak_fits):
        print(f"\n{len(weak_fits)} dataset(s) show a WEAKER fit - not necessarily wrong (small graphs "
              f"naturally fit worse), but worth a manual look before assuming the topology is clean:")
        print(weak_fits)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(power_law_df["dataset"].to_list(), power_law_df["r2"].to_list(), color="#4C72B0")
    ax.axvline(0.5, color="#C44E52", linestyle="--", linewidth=1, label="R\u00b2 = 0.5 reference line")
    ax.set_xlabel("Power-law fit R\u00b2 (higher = more heavy-tailed / real-transaction-shaped)")
    ax.set_title("Graph structure validity check across all ingested graph datasets")
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


**Reading this result**: on the reference run, **15 of 15 graph datasets showed a strong
heavy-tailed signature (R² ≥ 0.5)** — every single graph in the portfolio, from Elliptic's
anonymized Bitcoin transactions to IBM AML's synthetic bank transfers to XBlock's NFT transfers,
independently exhibits the same real-transaction-network signature. That's a stronger and more
specific result than "the pipeline didn't crash" — it's direct statistical evidence that every
domain in this portfolio produced structurally realistic graphs, not degenerate or accidentally
malformed ones, which is exactly what **TWP Regularization** needs: real structural diversity to
regularize across, not eight copies of noise.

## §3. Temporal signal audit — does Burst-Aware Temporal Decay have anything to decay?

Burst-Aware Temporal Decay is meaningless without real time-ordering on edges. This scans every
ingested dataset's edge schema for a timestamp-shaped column (exact name varies a lot across
16+ independently-sourced datasets — `Timestamp`, `step`, `time_step`, `blockNumber`, etc.) and
reports which datasets can actually support temporal modeling versus which are timestamp-free
and would need to be excluded from (or handled specially by) the temporal component.


In [ ]:
TIMESTAMP_HINTS = ["timestamp", "time", "date", "step", "time_step", "date_time",
                    "created_at", "block_number", "blockNumber"]

def find_timestamp_col(columns):
    cols_lower = {c.lower(): c for c in columns}
    for hint in TIMESTAMP_HINTS:
        if hint in cols_lower:
            return cols_lower[hint]
    return None

temporal_rows = []
for ds_dir in sorted(OUTPUT_DIR.iterdir()):
    if not ds_dir.is_dir():
        continue
    target = ds_dir / "edges.parquet"
    if not target.exists():
        target = ds_dir / "raw_table.parquet"
        if not target.exists():
            target = ds_dir / "labeled_transactions.parquet"
    if not target.exists():
        continue
    cols = pl.scan_parquet(target).collect_schema().names()
    ts_col = find_timestamp_col(cols)
    temporal_rows.append({"dataset": ds_dir.name, "has_temporal_signal": ts_col is not None,
                           "timestamp_column": ts_col})

temporal_df = pl.DataFrame(temporal_rows).sort("has_temporal_signal", descending=True)
n_temporal = temporal_df.filter(pl.col("has_temporal_signal")).height
print(f"{n_temporal} of {len(temporal_df)} datasets have a detected timestamp/time-order column.")
print("Datasets WITHOUT temporal signal (Burst-Aware Temporal Decay needs a fallback or exclusion for these):")
print(temporal_df.filter(~pl.col("has_temporal_signal"))["dataset"].to_list())
temporal_df


**Reading this result**: on the reference run, **14 of 22 datasets had a detected edge-level
timestamp column**. The 8 without it break into two genuinely different cases, worth telling
apart rather than treating as one blanket limitation:
1. **Datasets that are legitimately timestamp-free by design** — the IBM AML `*_accounts` lookup
   tables and Smart Ponzi's contract-label table were never transaction logs in the first place,
   so there was never temporal signal to have.
2. **Elliptic v1 and v2 — a genuinely important, non-obvious finding.** Elliptic's temporal
   signal (`time_step`) lives on **nodes**, not edges — the edgelist itself has no timestamp
   column. A Burst-Aware Temporal Decay implementation that only looks at edge timestamps will
   silently treat Elliptic as timestamp-free and get it wrong; the correct fix is to pull
   temporal signal from the node feature table for these two datasets specifically, not to
   exclude them from the temporal component. This is exactly the kind of dataset-specific
   handling this audit exists to surface before it becomes a silent Layer 2 bug.

## §4. IBM AML `Patterns.txt` deep-dive — real typology diversity, not one pattern repeated

This is the concrete evidence for the typology-diversity claim: how many distinct laundering
scheme *types* actually showed up (STACK, CYCLE, SCATTER-GATHER, etc.), across how many separate
labeled schemes, and how HI (high illicit ratio) compares to LI (low illicit ratio) tiers.


In [ ]:
pattern_dirs = [d for d in sorted(OUTPUT_DIR.iterdir())
                if d.is_dir() and d.name.startswith("ibm_amlsim_") and (d / "patterns.parquet").exists()]

if not pattern_dirs:
    print("No ibm_amlsim_* patterns.parquet found this run.")
else:
    all_patterns = []
    for d in pattern_dirs:
        df = pl.read_parquet(d / "patterns.parquet").with_columns(pl.lit(d.name).alias("tier"))
        all_patterns.append(df)
    patterns_all = pl.concat(all_patterns, how="diagonal_relaxed")

    type_summary = (patterns_all.group_by(["tier", "pattern_type"])
                     .agg(pl.col("pattern_id").n_unique().alias("n_schemes"),
                          pl.len().alias("n_labeled_transactions"))
                     .sort(["tier", "n_schemes"], descending=[False, True]))
    print(f"Total: {patterns_all['pattern_id'].n_unique():,} distinct laundering schemes across "
          f"{patterns_all['pattern_type'].n_unique()} pattern types, {len(pattern_dirs)} tiers.")
    type_summary


In [ ]:
if pattern_dirs:
    pivot = (type_summary.pivot(values="n_schemes", index="pattern_type", on="tier")
             .fill_null(0))
    fig, ax = plt.subplots(figsize=(11, 5))
    tiers = [c for c in pivot.columns if c != "pattern_type"]
    pattern_types = pivot["pattern_type"].to_list()
    x = np.arange(len(pattern_types))
    width = 0.8 / max(len(tiers), 1)
    for i, tier in enumerate(tiers):
        ax.bar(x + i * width, pivot[tier].to_list(), width, label=tier)
    ax.set_xticks(x + width * (len(tiers) - 1) / 2)
    ax.set_xticklabels(pattern_types, rotation=30, ha="right")
    ax.set_ylabel("Number of distinct schemes")
    ax.set_title("Laundering scheme types by IBM AML tier")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


**Reading this chart**: each bar cluster shows how many distinct laundering *schemes* of that
type exist per tier, not how many transactions — a STACK scheme might involve 6 transactions,
a long CYCLE might involve dozens, so the transaction-count total in the printed summary above
is a different (and usually much larger) number than the scheme count shown here. What matters
for the typology-validation claim is that **multiple genuinely distinct pattern types exist at
all**, and that they show up differently across HI and LI tiers — that's the evidence the model
can be tested on "does it recognize a CYCLE differently from a STACK," not just "does it flag a
large transaction," which is the whole point of using labeled typology data instead of a binary
fraud flag.

## §5. Feature dimensionality across datasets — the heterogeneity a HT-GNN actually needs

A Heterogeneous GNN assigns a separate embedding/projection per node or edge *type* precisely
because different types carry different numbers/kinds of features. This table is the direct
evidence for that heterogeneity: if every dataset had the same feature count, treating them as
different node/edge types would add complexity without benefit. Wide variation here is exactly
what justifies the "Heterogeneous" in HT-GNN.


In [ ]:
feature_dim_rows = []
for ds_dir in sorted(OUTPUT_DIR.iterdir()):
    if not ds_dir.is_dir():
        continue
    nodes_pq = ds_dir / "nodes.parquet"
    if nodes_pq.exists():
        cols = pl.scan_parquet(nodes_pq).collect_schema()
        numeric_feat_cols = [c for c, dt in cols.items() if dt in (pl.Float32, pl.Float64, pl.Int32, pl.Int64)
                             and c not in ("node_id", "label")]
        feature_dim_rows.append({"dataset": ds_dir.name, "n_node_feature_dims": len(numeric_feat_cols)})

feature_dim_df = pl.DataFrame(feature_dim_rows).sort("n_node_feature_dims", descending=True)
print(f"Node feature dimensionality ranges from {feature_dim_df['n_node_feature_dims'].min()} to "
      f"{feature_dim_df['n_node_feature_dims'].max()} across {len(feature_dim_df)} datasets - "
      f"this spread is the concrete justification for per-type embedding layers in the HT-GNN.")
feature_dim_df


**Reading this table**: on the reference run, node feature dimensionality ranged from **0 to
167** across 15 datasets — Elliptic v1's 165 anonymized features (plus `time_step` and `label`)
at the high end, down to datasets like `cc_transactions`, `mtgox_leaked`, `paysim1`,
`paysim_extended`, `saml_d`, and `xblock_eth` at 0 — meaning those datasets' real signal lives
entirely in **edge** attributes (amount, timestamp, transaction type), not node features, since
their "nodes" are just bare account/address identifiers with no inherent attributes of their own.
That's not a gap — it's an accurate reflection of what these datasets actually are, and it's
itself evidence for heterogeneity: a HT-GNN's node-type embedding for "Elliptic transaction" and
"PaySim account" need to be different-shaped layers, not the same one reused, because the
underlying data genuinely carries different information at the node level.

---
# Validity Conclusion — does this data support the four HT-GNN claims?

Pulling every section above together, with the actual reference-run findings against each claim:

| Component | Requirement | Finding on the reference run |
|---|---|---|
| **Burst-Aware Temporal Decay** | Real time-order signal on edges | §3: 14 of 22 datasets have edge-level temporal signal directly; 2 more (Elliptic v1/v2) carry it on nodes instead — usable, but requires the temporal component to read from the right table per dataset, not edges everywhere by default |
| **Task-Free Continual Learning** | Genuinely different domains, not variations on one theme | §1: confirmed — 8 distinct domains (crypto, DeFi, traditional banking, MFS, typology validation, crypto-crime, cross-domain benchmarks, own-tool) across 19 successfully ingested datasets |
| **TWP Regularization** | Enough distinct, structurally real "tasks" to regularize across | §2: 15 of 15 graph datasets showed a strong heavy-tailed signature (R² ≥ 0.5) — every domain independently produced structurally realistic graphs |
| **Conformal Prediction (MAPIE)** | Labeled data with a genuine, non-degenerate positive rate | Part A §A.3 — spot-check any dataset showing exactly 0% or 100% before calibrating on it |

**Bottom line for the methodology writeup**: this isn't a single pass/fail number — it's that
domain diversity (§1), real graph structure (§2, statistically confirmed, not just visually
plausible), available temporal signal in the large majority of datasets (§3, with the two
exceptions understood and explained, not silently wrong), genuine typology diversity in IBM AML
specifically (§4), and real feature-dimensionality heterogeneity (§5) are all independently
consistent with what the four mathematical components need. Where they aren't perfectly aligned —
Elliptic's node-level (not edge-level) timestamps, the two datasets with zero node features —
those are concrete, specific, already-understood things to account for explicitly in Layer 2's
design, not open questions.


---
# Layer 1 → Layer 2 Handoff

This is the practical section: what exists on disk right now, exactly how to load it, and the
specific decisions Layer 2 needs to make given everything found above — not a repeat of the EDA,
but the action items that follow from it.

## What's actually in `graph_data/`

Every successfully-ingested dataset produced one of these three shapes:

| Shape | Files | Datasets | What it means for Layer 2 |
|---|---|---|---|
| **Standard graph** | `nodes.parquet` + `edges.parquet` | Elliptic v1, DGraphFin, XBlock-ETH, PaySim (both), SAML-D, cc_transactions, Mt.Gox, eth_phishing, IBM AML tiers, data_generator | Load both, build a PyG `Data`/`HeteroData` object per dataset or per node/edge type |
| **Non-graph tabular** | `raw_table.parquet` | ULB, SynthAML, Smart Ponzi, IBM AML `*_accounts` | No entity linkage in the source data — use as auxiliary features/lookups, never force into graph construction |
| **Special-shaped** | `labeled_transactions.parquet` (eth_phishing_2nd), `background_*` + `patterns.parquet` (Elliptic v2, IBM AML) | eth_phishing_2nd, Elliptic v2, IBM AML tiers | See dataset-specific notes below |

## Dataset-specific notes Layer 2 needs to know

- **Elliptic v2**: `nodes.parquet`/`edges.parquet` are the small *labeled* subgraph.
  `background_nodes.parquet` (full features) and `background_edges_topology.parquet`
  (structure only, no features — see the earlier disk-budget explanation) are a **separate**,
  much larger context graph. Layer 2 needs to explicitly decide how to combine them — e.g. use
  the labeled subgraph for supervised loss and the background for structural context via a
  neighbor sampler, not merge them into one table.
- **IBM AML tiers**: `patterns.parquet` gives scheme-level typology labels (which specific
  transactions form which specific laundering scheme) — richer than the binary `Is Laundering`
  flag already in `edges.parquet`. Join on the transaction's identifying fields if scheme-level
  supervision is wanted, not just binary classification.
- **eth_phishing_2nd**: `labeled_transactions.parquet` carries `actor_type` (normal/phishing) and
  `hop_order` (first/second-order) columns — this is real category information, not just a flat
  transaction table; use it rather than discarding it.
- **Elliptic v1 and v2 temporal signal**: lives in the node feature table (`time_step`), not on
  edges — confirmed directly in §3 above. Any temporal-decay code that only reads edge timestamps
  will silently skip these two datasets; it needs a per-dataset branch.

## Concrete checklist before starting Layer 2 model code

1. Re-run this notebook's `ingestion_report`/run-report cell and confirm every dataset you plan
   to use shows `SUCCESS` — a silently-missing dataset produces an empty table, not an error, in
   most downstream code.
2. For any dataset flagged with a power-law R² below 0.5 in §2, do a manual spot-check before
   trusting its topology.
3. Decide the temporal-decay fallback for the datasets flagged in §3 as timestamp-free by design
   (not the Elliptic exception, which has a real fix) — synthetic ordering, or explicit exclusion.
4. Build the per-node/edge-type embedding layer dimensions directly from §5's feature-dimension
   table rather than hardcoding them — it's already computed and correct.
5. Decide the Elliptic v2 background-graph sampling strategy (fixed-fanout neighbor sampling is
   the standard approach for a graph this size) before writing the training loop, not during it.


---
# Universal Ingestion: Batch Files + Streaming/Live Data

Everything above handles static files that already exist in full on disk — that's genuinely a
different problem from live data, and it's worth being direct about what changes and what
doesn't, rather than blurring the two together.

## What's true about combining "batch" and "streaming" here

**A Kaggle notebook session cannot hold an always-on network listener open** — a real Kafka
consumer or websocket server needs a persistent process, and Kaggle sessions aren't designed to
run as services. So this section does NOT pretend to run a live Kafka/websocket consumer inside
the notebook. What it DOES do, genuinely:

1. **Extends file-format coverage** — JSON/JSONL, Excel, and Parquet passthrough, on top of the
   CSV/TXT/pickle/npz/pt formats already handled above, through the same auto-detecting src/dst/
   label engine.
2. **Adds an incremental ingestion core** (`ingest_stream_batch`) that accepts any small batch of
   new records — from a file, a list of dicts, a Kafka message, a websocket payload, anything —
   and appends it as a new, checkpointed part-file, without ever reprocessing or duplicating data
   already ingested. This is the actual mechanism that makes "streaming" meaningful: idempotent,
   incremental, resumable writes.
3. **Adds a working "watch a folder for new files" poller** — this is a real, legitimate
   near-real-time pattern (the same one many production systems use for an SFTP inbox or S3
   event before graduating to a full message queue), and it genuinely runs inside a notebook,
   either in a bounded loop here or via Kaggle's "Schedule notebook to run" feature for real
   periodic re-ingestion.
4. **Unifies the view for Layer 2** (`load_full_dataset`) — reads bulk-batch parquet AND any
   streaming part-files together as one dataset, so Layer 2 never needs to know or care whether
   a given row arrived via the original bulk load or a later live update.

**To graduate this to a true always-on stream later**: swap the watch-folder poller's body for a
real Kafka/websocket consumer loop that calls `ingest_stream_batch()` per message — that function
is already the correct, generic sink. Nothing else in the pipeline needs to change.


## Extended file-format support

Same auto-detecting engine as before, now covering JSON/JSONL, Excel, and Parquet passthrough.


In [ ]:
def load_json_records(dataset_name, path, src_hints=None, dst_hints=None, label_hints=None):
    """JSON array or JSONL (one record per line) - both read the same way via read_json_auto."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"file missing -> {path}")

    files_sql = build_files_sql([path])
    probe = f"read_json_auto({files_sql})"
    cur = con.execute(f"SELECT * FROM {probe} LIMIT 0")
    raw_cols = [d[0] for d in cur.description]
    stripped_cols = [c.strip() for c in raw_cols]

    src_hints = src_hints or SRC_HINTS
    dst_hints = dst_hints or DST_HINTS
    label_hints = label_hints or LABEL_HINTS
    src_col = next((c for c in stripped_cols if c in src_hints), None)
    dst_col = next((c for c in stripped_cols if c in dst_hints), None)
    label_col = next((c for c in stripped_cols if c in label_hints), None)

    out_dir = OUTPUT_DIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    if not src_col or not dst_col:
        con.execute(f"COPY (SELECT * FROM {probe}) TO '{out_dir / 'raw_table.parquet'}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        n = con.execute(f"SELECT COUNT(*) FROM '{out_dir / 'raw_table.parquet'}'").fetchone()[0]
        print(f"  Saved as flat table (no src/dst detected): {n:,} rows")
        return {"status": "tabular_fallback", "rows": n}

    exclude_set = {src_col, dst_col} | ({label_col} if label_col else set())
    exclude_sql = ", ".join(f'"{c}"' for c in exclude_set)
    label_expr = f'"{label_col}"' if label_col else "-1"

    con.execute(f"""
        COPY (SELECT CAST("{src_col}" AS VARCHAR) AS node_id, 'Entity' AS node_type
              FROM {probe} WHERE "{src_col}" IS NOT NULL
              UNION
              SELECT CAST("{dst_col}" AS VARCHAR) AS node_id, 'Entity' AS node_type
              FROM {probe} WHERE "{dst_col}" IS NOT NULL)
        TO '{out_dir / "nodes.parquet"}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    con.execute(f"""
        COPY (SELECT TRIM("{src_col}") AS src, TRIM("{dst_col}") AS dst, {label_expr} AS label,
                     'transaction' AS edge_type, * EXCLUDE ({exclude_sql})
              FROM {probe} WHERE "{src_col}" IS NOT NULL AND "{dst_col}" IS NOT NULL)
        TO '{out_dir / "edges.parquet"}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    n_nodes = con.execute(f"SELECT COUNT(*) FROM '{out_dir / 'nodes.parquet'}'").fetchone()[0]
    n_edges = con.execute(f"SELECT COUNT(*) FROM '{out_dir / 'edges.parquet'}'").fetchone()[0]
    print(f"  Saved: {n_nodes:,} nodes | {n_edges:,} edges")
    return {"status": "ok", "n_nodes": n_nodes, "n_edges": n_edges}


def load_excel_file(dataset_name, path, sheet_name=None, src_hints=None, dst_hints=None, label_hints=None):
    """Excel (.xlsx/.xls). DuckDB has no native Excel reader, so this loads via Polars
    (needs the fastexcel package) then hands off to DuckDB for the same standardized
    node/edge construction as every other loader."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"file missing -> {path}")

    df = pl.read_excel(path, sheet_name=sheet_name) if sheet_name else pl.read_excel(path)
    df.columns = [c.strip() for c in df.columns]
    con.register("excel_tmp", df.to_pandas())

    src_hints = src_hints or SRC_HINTS
    dst_hints = dst_hints or DST_HINTS
    label_hints = label_hints or LABEL_HINTS
    src_col = next((c for c in df.columns if c in src_hints), None)
    dst_col = next((c for c in df.columns if c in dst_hints), None)
    label_col = next((c for c in df.columns if c in label_hints), None)

    out_dir = OUTPUT_DIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    if not src_col or not dst_col:
        con.execute(f"COPY (SELECT * FROM excel_tmp) TO '{out_dir / 'raw_table.parquet'}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        n = len(df)
        print(f"  Saved as flat table (no src/dst detected): {n:,} rows")
        con.unregister("excel_tmp")
        return {"status": "tabular_fallback", "rows": n}

    exclude_set = {src_col, dst_col} | ({label_col} if label_col else set())
    exclude_sql = ", ".join(f'"{c}"' for c in exclude_set)
    label_expr = f'"{label_col}"' if label_col else "-1"

    con.execute(f"""
        COPY (SELECT CAST("{src_col}" AS VARCHAR) AS node_id, 'Entity' AS node_type
              FROM excel_tmp WHERE "{src_col}" IS NOT NULL
              UNION
              SELECT CAST("{dst_col}" AS VARCHAR) AS node_id, 'Entity' AS node_type
              FROM excel_tmp WHERE "{dst_col}" IS NOT NULL)
        TO '{out_dir / "nodes.parquet"}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    con.execute(f"""
        COPY (SELECT TRIM("{src_col}") AS src, TRIM("{dst_col}") AS dst, {label_expr} AS label,
                     'transaction' AS edge_type, * EXCLUDE ({exclude_sql})
              FROM excel_tmp WHERE "{src_col}" IS NOT NULL AND "{dst_col}" IS NOT NULL)
        TO '{out_dir / "edges.parquet"}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    n_nodes = con.execute(f"SELECT COUNT(*) FROM '{out_dir / 'nodes.parquet'}'").fetchone()[0]
    n_edges = con.execute(f"SELECT COUNT(*) FROM '{out_dir / 'edges.parquet'}'").fetchone()[0]
    print(f"  Saved: {n_nodes:,} nodes | {n_edges:,} edges")
    con.unregister("excel_tmp")
    return {"status": "ok", "n_nodes": n_nodes, "n_edges": n_edges}


def load_parquet_passthrough(dataset_name, path):
    """Already-Parquet input - just validate and copy into the standard output location
    rather than re-deriving nodes/edges (respects whatever structure it already has)."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"file missing -> {path}")
    out_dir = OUTPUT_DIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "raw_table.parquet"
    con.execute(f"COPY (SELECT * FROM read_parquet('{path}')) TO '{out_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    n = con.execute(f"SELECT COUNT(*) FROM '{out_path}'").fetchone()[0]
    print(f"  Copied through: {n:,} rows")
    return {"status": "ok", "rows": n}


## Streaming / incremental ingestion core

`ingest_stream_batch` is the generic sink every live source feeds into — file-tail, Kafka
message, websocket payload, anything that can be turned into a small batch of records. It's
checkpointed by `batch_id`, so calling it twice with the same `batch_id` is a safe no-op, not a
duplicate write (verified directly below with an idempotency test, not just a claim).


In [ ]:
import re, hashlib
from datetime import datetime, timezone

CHECKPOINT_PATH = OUTPUT_DIR / "_checkpoints.parquet"

def _load_checkpoints():
    if CHECKPOINT_PATH.exists():
        return pl.read_parquet(CHECKPOINT_PATH)
    return pl.DataFrame(schema={"dataset_name": pl.Utf8, "source_id": pl.Utf8,
                                 "ingested_at": pl.Utf8, "rows_ingested": pl.Int64})

def _save_checkpoint(dataset_name, source_id, rows_ingested):
    df = _load_checkpoints()
    now = datetime.now(timezone.utc).isoformat()
    df = df.filter(~((pl.col("dataset_name") == dataset_name) & (pl.col("source_id") == source_id)))
    new_row = pl.DataFrame({"dataset_name": [dataset_name], "source_id": [source_id],
                             "ingested_at": [now], "rows_ingested": [rows_ingested]})
    df = pl.concat([df, new_row])
    df.write_parquet(CHECKPOINT_PATH)

def _is_already_ingested(dataset_name, source_id):
    df = _load_checkpoints()
    if len(df) == 0:
        return False
    match = df.filter((pl.col("dataset_name") == dataset_name) & (pl.col("source_id") == source_id))
    return len(match) > 0


def ingest_stream_batch(dataset_name, records, batch_id=None):
    """
    Appends a small batch of new records (list of dicts, or a Polars DataFrame) as a new,
    checkpointed part-file under graph_data/<dataset_name>/streaming/. Safe to call repeatedly
    with the same batch_id - already-seen batches are skipped, not duplicated.
    """
    is_df = isinstance(records, pl.DataFrame)
    if (is_df and records.is_empty()) or (not is_df and not records):
        return {"status": "empty_batch"}

    batch_id = batch_id or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
    if _is_already_ingested(dataset_name, batch_id):
        return {"status": "already_ingested", "batch_id": batch_id}

    df = records if is_df else pl.DataFrame(records)
    stream_dir = OUTPUT_DIR / dataset_name / "streaming"
    stream_dir.mkdir(parents=True, exist_ok=True)

    # batch_id might be a full file path, a Kafka "topic:partition:offset" string, etc. -
    # sanitize for use as a filename while keeping the original for checkpoint tracking.
    safe_id = re.sub(r"[^A-Za-z0-9_-]", "_", batch_id)[:100]
    if safe_id != batch_id:
        safe_id += "_" + hashlib.md5(batch_id.encode()).hexdigest()[:8]
    part_path = stream_dir / f"part-{safe_id}.parquet"

    df.write_parquet(part_path, compression="zstd")
    _save_checkpoint(dataset_name, batch_id, len(df))
    return {"status": "ok", "rows": len(df), "path": str(part_path), "batch_id": batch_id}


def load_full_dataset(dataset_name):
    """Unifies bulk-batch parquet + any streaming part-files into ONE view for Layer 2 -
    it never needs to know whether a row arrived via the original bulk load or a later
    live update."""
    out_dir = OUTPUT_DIR / dataset_name
    if not out_dir.exists():
        return None
    patterns = []
    for fname in ["nodes.parquet", "edges.parquet", "raw_table.parquet", "transactions.parquet"]:
        if (out_dir / fname).exists():
            patterns.append(str(out_dir / fname))
    stream_dir = out_dir / "streaming"
    if stream_dir.exists() and list(stream_dir.glob("*.parquet")):
        patterns.append(str(stream_dir / "*.parquet"))
    if not patterns:
        return None
    return con.execute(f"SELECT * FROM read_parquet({patterns}, union_by_name=true)").pl()


def watch_folder_and_ingest(dataset_name, folder, max_polls=3, poll_interval_s=5):
    """
    Genuine, working near-real-time ingestion for a notebook context: repeatedly checks
    `folder` for files not yet in the checkpoint store and ingests any new ones.

    WHAT THIS CAN AND CANNOT DO: this can run for a bounded number of polls interactively (as
    below), or be wired up via Kaggle's "Schedule notebook to run" feature for genuine periodic
    re-ingestion. It cannot hold an always-on network listener open in this session. To graduate
    to a true always-on stream, replace this function's loop body with a Kafka/websocket
    consumer that calls ingest_stream_batch() per message instead - everything downstream
    (checkpointing, load_full_dataset) stays exactly the same.
    """
    folder = Path(folder)
    poll = 0
    total_new_rows = 0
    while poll < max_polls:
        poll += 1
        current_files = sorted(folder.glob("*.csv")) if folder.exists() else []
        new_files = [f for f in current_files if not _is_already_ingested(dataset_name, str(f))]
        if new_files:
            print(f"  [poll {poll}] {len(new_files)} new file(s): {[f.name for f in new_files]}")
            for f in new_files:
                df = pl.read_csv(f, infer_schema_length=10_000)
                result = ingest_stream_batch(dataset_name, df, batch_id=str(f))
                total_new_rows += result.get("rows", 0)
        else:
            print(f"  [poll {poll}] no new files")
        if poll < max_polls:
            time.sleep(poll_interval_s)
    print(f"  Watch loop ended after {poll} poll(s), {total_new_rows:,} new rows ingested total.")
    return total_new_rows


## Proof it actually works: simulated live feed + idempotency check

This runs a synthetic live feed through `ingest_stream_batch` across several "ticks," confirms
`load_full_dataset` sees everything unified, then re-runs the exact same feed and confirms
nothing gets duplicated. Swap `simulate_live_feed` for a real Kafka/websocket consumer later —
everything downstream is unchanged.


In [ ]:
def simulate_live_feed(n_batches=5, records_per_batch=3):
    """Stand-in for a real live source (Kafka consumer, websocket handler, API poller).
    Replace this generator with a real one to go from demo to production - ingest_stream_batch
    and everything downstream doesn't change."""
    for i in range(n_batches):
        batch = [{"src": f"user{i}", "dst": f"merchant{i}", "amount": 50.0 + i}
                 for _ in range(records_per_batch)]
        yield f"tick-{i}", batch

print("=== Simulated live feed: first run ===")
for batch_id, batch in simulate_live_feed():
    result = ingest_stream_batch("live_demo", batch, batch_id=batch_id)
    print(f"  {batch_id}: {result['status']}, rows={result.get('rows')}")

unified = load_full_dataset("live_demo")
print(f"\nUnified view after first run: {len(unified):,} rows")

print("\n=== Re-running the SAME feed (should all be skipped, not duplicated) ===")
for batch_id, batch in simulate_live_feed():
    result = ingest_stream_batch("live_demo", batch, batch_id=batch_id)
    print(f"  {batch_id}: {result['status']}")

unified2 = load_full_dataset("live_demo")
print(f"\nUnified view after re-run: {len(unified2):,} rows (should be unchanged)")
assert len(unified2) == len(unified), "BUG: re-run caused duplication"
print("Confirmed: no duplication on re-run.")
